# DeepEntXAI — Consolidated Code

**Explainable multimodal deep learning for anti-*Enterobacteriaceae* bioactivity.**
Authors: **Nagmi Bano, Dr Shaban Ahmad, Prof Khalid Raza** — Computational Intelligence and Bioinformatics Lab, Department of Computer Science, Jamia Millia Islamia, New Delhi, India.

This single notebook collects the **entire, leakage-free pipeline** in one place:
the engine package (`deepentxai/`) followed by the numbered pipeline stages
(`01`–`15`). Each section reproduces one source module verbatim. The dataset is
`Data_DeepEntXAI.xlsx` (49,093 activity-gap compounds).

Run order (as an installed pipeline): stages 01 → 08 (core) then 09 → 15 (extras).
Requires: `numpy pandas scikit-learn tensorflow-cpu keras xgboost rdkit optuna shap lime`.

---


# 1 · Engine package — `deepentxai/`
Shared library: configuration, molecular standardisation, labelling, feature generation, selection, scaffold splits, the CNN-LSTM model, training, evaluation, explainability and scoring.

## `deepentxai/config.py`


In [ ]:
"""Typed configuration for DEEPENTXAI_FINAL — numbered, self-contained layout.

Layout (root = DeepEntXAI_Final):
    01_Code/{config.yaml, thresholds.yaml, deepentxai/*, 00..07 numbered stages}
    02_Data/{01_Raw, 02_Processed, 03_Features_Raw, 04_Features_Selected, 05_Splits}
    03_Results/{01_Figures, 02_Metrics, 03_Model, 04_Explainability,
                05_Predictions, 06_Logs, 07_Optuna, 08_Rankings}

Every directory carries a numeric prefix so the pipeline order is unambiguous.
The engine scripts refer to directories only through the `dir_*` properties below,
so re-homing the whole project is a matter of editing this one file.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Any, Dict

import yaml


def _root() -> str:
    # this file: <root>/01_Code/deepentxai/config.py  ->  root is 3 levels up
    here = os.path.abspath(__file__)
    return os.path.dirname(os.path.dirname(os.path.dirname(here)))


@dataclass
class Config:
    raw: Dict[str, Any] = field(default_factory=dict)
    root: str = field(default_factory=_root)
    variant: str = ""          # retained for API compatibility; unused in the final layout

    @classmethod
    def load(cls, path: str | None = None, variant: str | None = None) -> "Config":
        root = _root()
        path = path or os.environ.get("DEEPENT_CONFIG") or os.path.join(root, "01_Code", "config.yaml")
        if not os.path.exists(path):
            raise FileNotFoundError(f"config not found: {path}")
        with open(path) as fh:
            return cls(raw=yaml.safe_load(fh), root=root, variant="")

    def __getitem__(self, k: str) -> Any: return self.raw[k]
    def get(self, k: str, d: Any = None) -> Any: return self.raw.get(k, d)

    @property
    def seed(self) -> int: return int(self.raw.get("seed", 42))

    def _mk(self, *parts: str) -> str:
        p = os.path.join(self.root, *parts)
        os.makedirs(p, exist_ok=True)
        return p + os.sep

    # --- 02_Data ---
    @property
    def dir_raw(self) -> str:
        override = os.environ.get("DEEPENT_RAW_DIR")
        if override:
            os.makedirs(override, exist_ok=True)
            return override + os.sep
        return self._mk("02_Data", "01_Raw")
    @property
    def dir_processed(self) -> str: return self._mk("02_Data", "02_Processed")
    @property
    def dir_features_raw(self) -> str: return self._mk("02_Data", "03_Features_Raw")
    @property
    def dir_features_selected(self) -> str: return self._mk("02_Data", "04_Features_Selected")
    @property
    def dir_splits(self) -> str: return self._mk("02_Data", "05_Splits")

    # --- 03_Results ---
    @property
    def dir_figures(self) -> str: return self._mk("03_Results", "01_Figures")
    @property
    def dir_metrics(self) -> str: return self._mk("03_Results", "02_Metrics")
    @property
    def dir_reports(self) -> str: return self._mk("03_Results", "02_Metrics")
    @property
    def dir_models(self) -> str: return self._mk("03_Results", "03_Model")
    @property
    def dir_explain(self) -> str: return self._mk("03_Results", "04_Explainability")
    @property
    def dir_predictions(self) -> str: return self._mk("03_Results", "05_Predictions")
    @property
    def dir_logs(self) -> str: return self._mk("03_Results", "06_Logs")
    @property
    def dir_optuna(self) -> str: return self._mk("03_Results", "07_Optuna")
    @property
    def dir_rankings(self) -> str: return self._mk("03_Results", "08_Rankings")


## `deepentxai/utils.py`


In [ ]:
"""Shared utilities: reproducible seeding, structured logging, small IO helpers."""
from __future__ import annotations

import json
import logging
import os
import random
from typing import Any, Dict

import numpy as np


def set_seed(seed: int = 42) -> None:
    """Seed every RNG that can affect a DEEPENTXAI result."""
    os.environ.setdefault("PYTHONHASHSEED", str(seed))
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except Exception:
        pass
    try:
        import torch
        torch.manual_seed(seed)
    except Exception:
        pass


def get_logger(name: str, log_dir: str | None = None, level: int = logging.INFO) -> logging.Logger:
    """Dual (console + file) structured logger; safe to call repeatedly."""
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if logger.handlers:                      # avoid duplicate handlers in notebooks
        return logger
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                            "%H:%M:%S")
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
        fh = logging.FileHandler(os.path.join(log_dir, f"{name}.log"), mode="a")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger


def detect_gpu() -> bool:
    """True if TensorFlow can see a GPU (drives mixed-precision decisions)."""
    try:
        import tensorflow as tf
        return len(tf.config.list_physical_devices("GPU")) > 0
    except Exception:
        return False


def save_json(obj: Dict[str, Any], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=str)


## `deepentxai/standardize.py`


In [ ]:
"""Professional cheminformatics standardisation of molecular structures.

Canonicalise SMILES, strip salts (largest fragment), neutralise charges, remove
mixtures / invalid molecules, and apply size filters. Every rejection reason is
counted so a reproducible preprocessing report can be generated.
"""
from __future__ import annotations

from collections import Counter
from typing import Optional, Tuple

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")


class MoleculeStandardizer:
    """Convert an input SMILES to a clean, canonical, neutral parent structure."""

    def __init__(self, cfg: dict):
        self.largest_fragment = cfg.get("largest_fragment", True)
        self.neutralize = cfg.get("neutralize", True)
        self.remove_stereo = cfg.get("remove_stereo", False)
        self.min_heavy = int(cfg.get("min_heavy_atoms", 3))
        self.max_heavy = int(cfg.get("max_heavy_atoms", 100))
        self._lfc = rdMolStandardize.LargestFragmentChooser()
        self._unch = rdMolStandardize.Uncharger()
        self.reasons: Counter = Counter()

    def __call__(self, smiles: str) -> Tuple[Optional[str], Optional[float]]:
        m = Chem.MolFromSmiles(str(smiles))
        if m is None:
            self.reasons["invalid_smiles"] += 1
            return None, None
        try:
            if self.largest_fragment:
                m = self._lfc.choose(m)          # remove salts / counter-ions / mixtures
            if self.neutralize:
                m = self._unch.uncharge(m)
            if self.remove_stereo:
                Chem.RemoveStereochemistry(m)
        except Exception:
            self.reasons["standardize_error"] += 1
            return None, None
        n = m.GetNumHeavyAtoms()
        if n < self.min_heavy:
            self.reasons["too_small"] += 1
            return None, None
        if n > self.max_heavy:
            self.reasons["too_large"] += 1
            return None, None
        self.reasons["kept"] += 1
        return Chem.MolToSmiles(m), Descriptors.MolWt(m)

    def report(self) -> dict:
        return dict(self.reasons)


## `deepentxai/labeling.py`


In [ ]:
"""Automatic, configurable binary activity labelling from experimental values.

Default: Active if IC50/EC50/Ki/Kd <= 10 uM (configurable). MIC is handled on its
own ug/mL scale. Ambiguous records are dropped. Per-structure conflicts are
resolved by majority vote (ties dropped), yielding one binary label per compound.
"""
from __future__ import annotations

from typing import Optional

import numpy as np
import pandas as pd

# unit -> factor to micromolar (uM)
_TO_UM = {"nM": 1e-3, "uM": 1.0, "µM": 1.0, "mM": 1e3, "M": 1e6, "pM": 1e-6, "fM": 1e-9}
_AFFINITY = {"IC50", "EC50", "Ki", "Kd"}


class ActivityLabeler:
    """Turn (type, value, units, MW) rows into a binary Active/Inactive label."""

    def __init__(self, cfg: dict):
        self.thr_uM = float(cfg["affinity_threshold_uM"])
        self.inactive_uM = float(cfg.get("affinity_inactive_uM", cfg["affinity_threshold_uM"]))
        self.mic_active = float(cfg["mic_active_ugml"])
        self.mic_inactive = float(cfg["mic_inactive_ugml"])
        self.drop_ambiguous = bool(cfg.get("drop_ambiguous", True))
        self.conflict_policy = cfg.get("conflict_policy", "majority")

    # -- per-measurement label --------------------------------------------
    def label_one(self, typ: str, value, units, mw: Optional[float]) -> Optional[int]:
        try:
            v = float(value)
        except (TypeError, ValueError):
            return None
        if v <= 0:
            return None
        if typ in _AFFINITY:
            if units not in _TO_UM:
                return None
            uM = v * _TO_UM[units]
            if uM <= self.thr_uM:
                return 1
            if uM > self.inactive_uM:
                return 0
            return None
        if typ == "MIC":
            if units in ("ug.mL-1", "ug/mL"):
                ugml = v
            elif units in _TO_UM and mw:                 # molar MIC -> ug/mL
                ugml = (v * _TO_UM[units] * 1e-6) * mw * 1e6
            else:
                return None
            if ugml <= self.mic_active:
                return 1
            if ugml >= self.mic_inactive:
                return 0
            return None
        return None

    # -- aggregate to one label per structure -----------------------------
    def resolve(self, df: pd.DataFrame, smiles_col: str = "smiles", label_col: str = "lab") -> pd.DataFrame:
        """Majority vote per unique structure; drop 50/50 ties."""
        g = df.groupby(smiles_col)[label_col]
        agg = g.mean().reset_index(name="frac_active")
        agg["n_measurements"] = g.count().values
        if self.conflict_policy == "majority":
            agg = agg[agg["frac_active"] != 0.5]
        agg["label"] = (agg["frac_active"] > 0.5).astype(int)
        return agg[[smiles_col, "label", "n_measurements", "frac_active"]]


## `deepentxai/download.py`


In [ ]:
"""Automated, reproducible bioactivity download from ChEMBL and PubChem BioAssay.

Uses the official public REST APIs. Results are cached under ``data/raw`` so a
re-run reuses them (``download.reuse_cache``) instead of re-hitting the network.
Only quantitative / outcome-bearing records with a SMILES are retained.
"""
from __future__ import annotations

import json
import os
import time
import urllib.parse
import urllib.request
from typing import Dict, List

import pandas as pd

from .config import Config
from .utils import get_logger

CHEMBL = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUG = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
_UA = {"User-Agent": "DEEPENTXAI/1.0 (research)"}


def _get(url: str, timeout: int = 60, retries: int = 3):
    for k in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=_UA), timeout=timeout) as r:
                return json.load(r)
        except Exception:
            if k == retries - 1:
                return None
            time.sleep(1.0)
    return None


class ChEMBLDownloader:
    """Pull IC50/EC50/Ki/Kd/MIC activities for the configured organisms."""

    def __init__(self, cfg: Config, log):
        self.cfg, self.log = cfg, log

    def run(self) -> pd.DataFrame:
        dl = self.cfg["download"]
        organisms = self.cfg["target"]["organisms"]
        types = dl["chembl_activity_types"]
        max_pages = int(dl["chembl_max_pages_per_query"])
        rows: List[Dict] = []
        for org in organisms:
            n0 = len(rows)
            for typ in types:
                offset = 0
                for _ in range(max_pages):
                    q = urllib.parse.urlencode({"target_organism": org, "standard_type": typ,
                                                "limit": 1000, "offset": offset})
                    d = _get(f"{CHEMBL}?{q}")
                    if not d:
                        break
                    acts = d.get("activities", [])
                    if not acts:
                        break
                    for a in acts:
                        smi = a.get("canonical_smiles")
                        val = a.get("standard_value")
                        if smi and val is not None and a.get("standard_units"):
                            rows.append({
                                "smiles": smi, "value": val, "units": a.get("standard_units"),
                                "type": typ, "relation": a.get("standard_relation"),
                                "chembl_id": a.get("molecule_chembl_id"),
                                "assay_id": a.get("assay_chembl_id"),
                                "organism": org, "source": "ChEMBL"})
                    offset += 1000
                    if d["page_meta"].get("next") is None:
                        break
            self.log.info(f"ChEMBL {org}: +{len(rows) - n0} (total {len(rows)})")
        return pd.DataFrame(rows)


class PubChemDownloader:
    """Pull ACTIVE/INACTIVE compounds from Enterobacteriaceae PubChem bioassays."""

    def __init__(self, cfg: Config, log):
        self.cfg, self.log = cfg, log

    def _aids(self, org: str, n: int) -> List[str]:
        q = urllib.parse.urlencode({"db": "pcassay", "term": f'"{org}"[Target Organism]',
                                    "retmax": n, "retmode": "json", "sort": "relevance"})
        d = _get(f"{EUTILS}?{q}")
        return (d or {}).get("esearchresult", {}).get("idlist", [])

    def _cids(self, aid: str, kind: str) -> List[int]:
        d = _get(f"{PUG}/assay/aid/{aid}/cids/JSON?cids_type={kind}")
        info = (d or {}).get("InformationList", {}).get("Information", [{}])
        return info[0].get("CID", []) if info else []

    def _smiles(self, cids: List[int]) -> Dict[int, str]:
        out: Dict[int, str] = {}
        for i in range(0, len(cids), 100):
            chunk = cids[i:i + 100]
            d = _get(f"{PUG}/compound/cid/{','.join(map(str, chunk))}/property/SMILES/JSON")
            for p in (d or {}).get("PropertyTable", {}).get("Properties", []):
                smi = p.get("SMILES") or p.get("ConnectivitySMILES")
                if smi:
                    out[p["CID"]] = smi
            time.sleep(0.25)
        return out

    def run(self) -> pd.DataFrame:
        dl = self.cfg["download"]
        rows: List[Dict] = []
        seen = set()
        n_org = int(dl.get("pubchem_max_organisms", len(self.cfg["target"]["organisms"])))
        for org in self.cfg["target"]["organisms"][:n_org]:
            for aid in self._aids(org, int(dl["pubchem_max_assays_per_organism"])):
                if aid in seen or len(rows) >= int(dl["pubchem_max_total"]):
                    continue
                seen.add(aid)
                act = self._cids(aid, "active")
                ina = self._cids(aid, "inactive")[: int(dl["pubchem_max_inactive_per_assay"])]
                pairs = [(c, 1) for c in act] + [(c, 0) for c in ina]
                smap = self._smiles([c for c, _ in pairs])
                for cid, lab in pairs:
                    if cid in smap:
                        rows.append({"smiles": smap[cid], "label": lab, "cid": cid,
                                     "aid": aid, "organism": org, "source": "PubChem"})
            self.log.info(f"PubChem {org}: total {len(rows)}")
        return pd.DataFrame(rows).drop_duplicates(subset=["smiles", "label"])


class DataDownloader:
    """Orchestrate ChEMBL + PubChem download with caching under data/raw."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.log = get_logger("download", cfg.dir_logs)

    def _cached_or(self, name: str, builder) -> pd.DataFrame:
        path = os.path.join(self.cfg.dir_raw, name)
        if self.cfg["download"].get("reuse_cache", True) and os.path.exists(path):
            self.log.info(f"reusing cached {name}")
            return pd.read_csv(path)
        self.log.info(f"downloading {name} from API ...")
        df = builder()
        df.to_csv(path, index=False)
        self.log.info(f"saved {name}: {len(df)} rows")
        return df

    def download(self) -> Dict[str, pd.DataFrame]:
        out = {}
        if "ChEMBL" in self.cfg["download"]["sources"]:
            out["ChEMBL"] = self._cached_or("chembl_raw.csv", lambda: ChEMBLDownloader(self.cfg, self.log).run())
        if "PubChem" in self.cfg["download"]["sources"]:
            out["PubChem"] = self._cached_or("pubchem_raw.csv", lambda: PubChemDownloader(self.cfg, self.log).run())
        return out


## `deepentxai/features.py`


In [ ]:
"""Molecular feature generation: Morgan, RDKit descriptors, MACCS, ChemBERTa.

Each representation is computed directly from a (standardised) SMILES string.
ChemBERTa embeddings are produced by a pretrained molecular transformer and cached,
since they are the most expensive step.
"""
from __future__ import annotations

from typing import List, Tuple

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Descriptors import MoleculeDescriptors

RDLogger.DisableLog("rdApp.*")

RDKIT_NAMES: List[str] = [d[0] for d in Descriptors.descList]
_RDKIT_CALC = MoleculeDescriptors.MolecularDescriptorCalculator(RDKIT_NAMES)


class FeatureGenerator:
    """Compute the four molecular representations for a list of SMILES."""

    def __init__(self, cfg: dict):
        self.bits = int(cfg["morgan_bits"])
        self.radius = int(cfg["morgan_radius"])
        self.chemberta_model = cfg["chemberta_model"]
        self.max_len = int(cfg["chemberta_max_len"])

    # -- per-molecule fingerprints/descriptors ----------------------------
    def morgan(self, mol) -> np.ndarray:
        bv = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.bits)
        a = np.zeros((self.bits,), np.float32); DataStructs.ConvertToNumpyArray(bv, a); return a

    def maccs(self, mol) -> np.ndarray:
        bv = MACCSkeys.GenMACCSKeys(mol)
        a = np.zeros((167,), np.float32); DataStructs.ConvertToNumpyArray(bv, a); return a

    def rdkit(self, mol) -> np.ndarray:
        return np.array(_RDKIT_CALC.CalcDescriptors(mol), dtype=np.float64)

    @staticmethod
    def scaffold(mol) -> str:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

    # -- ChemBERTa embeddings (batched) -----------------------------------
    def chemberta(self, smiles: List[str], batch: int = 64, log=None) -> np.ndarray:
        import torch
        from transformers import AutoTokenizer, AutoModel
        tok = AutoTokenizer.from_pretrained(self.chemberta_model)
        mod = AutoModel.from_pretrained(self.chemberta_model).eval()
        out = []
        for i in range(0, len(smiles), batch):
            enc = tok(smiles[i:i + batch], padding=True, truncation=True,
                      max_length=self.max_len, return_tensors="pt")
            with torch.no_grad():
                out.append(mod(**enc).last_hidden_state[:, 0, :].numpy())   # CLS token
            if log and i % (batch * 30) == 0:
                log.info(f"  ChemBERTa {i}/{len(smiles)}")
        return np.vstack(out).astype(np.float32)

    # -- full block over a dataframe --------------------------------------
    def generate(self, smiles: List[str], log=None) -> Tuple[dict, np.ndarray]:
        mols = [Chem.MolFromSmiles(s) for s in smiles]
        keep = [i for i, m in enumerate(mols) if m is not None]
        mols = [mols[i] for i in keep]; kept_smiles = [smiles[i] for i in keep]
        if log:
            log.info(f"featurising {len(mols)} molecules")
        feats = {
            "X_morgan": np.stack([self.morgan(m) for m in mols]).astype(np.float32),
            "X_maccs": np.stack([self.maccs(m) for m in mols]).astype(np.float32),
            "X_rdkit": np.array([self.rdkit(m) for m in mols], dtype=np.float64),
            "scaffold": np.array([self.scaffold(m) for m in mols]),
        }
        feats["X_rdkit"] = np.where(np.isfinite(feats["X_rdkit"]), feats["X_rdkit"], np.nan).astype(np.float32)
        if log:
            log.info("computing ChemBERTa embeddings ...")
        feats["X_chemberta"] = self.chemberta(kept_smiles, log=log)
        return feats, np.array(keep)


## `deepentxai/selection.py`


In [ ]:
"""Leakage-free numerical-descriptor selection for the RDKit descriptor block.

Pipeline (fit on TRAIN only, then applied to every split):
    median impute -> variance filter -> correlation filter (|r|>thr)
    -> Mutual Information pre-ranking -> Recursive Feature Elimination -> scale.
The fitted transformer + selected feature names are persisted so the exact same
descriptors are reproduced at prediction time.
"""
from __future__ import annotations

from typing import List

import numpy as np
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


class FeatureSelector:
    def __init__(self, cfg: dict, feature_names: List[str], seed: int = 42):
        self.var_thr = float(cfg["features"]["variance_threshold"])
        self.corr_thr = float(cfg["features"]["correlation_threshold"])
        self.n_select = int(cfg["features"]["n_select"])
        self.names = list(feature_names)
        self.seed = seed
        self.imputer_: SimpleImputer | None = None
        self.keep_idx_: np.ndarray | None = None      # indices into original columns
        self.scaler_: StandardScaler | None = None
        self.selected_names_: List[str] = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "FeatureSelector":
        self.imputer_ = SimpleImputer(strategy="median").fit(X)
        Xi = self.imputer_.transform(X)

        # 1) variance filter
        var = Xi.var(axis=0)
        vmask = var > self.var_thr
        idx = np.where(vmask)[0]

        # 2) correlation filter (greedy drop of the later of any |r|>thr pair)
        Xc = Xi[:, idx]
        corr = np.corrcoef(Xc, rowvar=False)
        corr = np.nan_to_num(corr)
        drop = set()
        for i in range(corr.shape[0]):
            if i in drop:
                continue
            for j in range(i + 1, corr.shape[0]):
                if j not in drop and abs(corr[i, j]) > self.corr_thr:
                    drop.add(j)
        idx = idx[[k for k in range(len(idx)) if k not in drop]]

        # 3) Mutual Information pre-ranking -> top 2*n_select
        Xm = Xi[:, idx]
        mi = mutual_info_classif(Xm, y, random_state=self.seed)
        top = np.argsort(mi)[::-1][: max(self.n_select * 2, self.n_select)]
        idx = idx[top]

        # 4) RFE with a fast linear estimator down to n_select
        k = min(self.n_select, idx.shape[0])
        Xr = Xi[:, idx]
        rfe = RFE(LogisticRegression(max_iter=1000, class_weight="balanced"),
                  n_features_to_select=k, step=0.1).fit(Xr, y)
        idx = idx[rfe.support_]

        self.keep_idx_ = np.array(sorted(idx))
        self.selected_names_ = [self.names[i] for i in self.keep_idx_]
        self.scaler_ = StandardScaler().fit(Xi[:, self.keep_idx_])
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        Xi = self.imputer_.transform(X)
        return self.scaler_.transform(Xi[:, self.keep_idx_]).astype(np.float32)


## `deepentxai/splits.py`


In [ ]:
"""Leakage-free data splitting.

Produces an independent hold-out test set and Stratified K-Fold indices on the
remaining train pool. With ``strategy='scaffold'`` whole Bemis-Murcko scaffolds
are kept disjoint between train and test (and across CV folds), so structural
analogues never leak across the evaluation boundary — the key requirement for a
credible QSAR benchmark.
"""
from __future__ import annotations

from collections import defaultdict
from typing import Dict, List

import numpy as np
from sklearn.model_selection import StratifiedKFold


class DataSplitter:
    def __init__(self, cfg: dict, seed: int = 42):
        self.strategy = cfg["split"]["strategy"]
        self.test_size = float(cfg["split"]["test_size"])
        self.n_folds = int(cfg["split"]["n_folds"])
        self.seed = seed

    # -- scaffold-disjoint hold-out ---------------------------------------
    def _scaffold_holdout(self, scaffolds: np.ndarray, y: np.ndarray):
        groups = defaultdict(list)
        for i, s in enumerate(scaffolds):
            groups[s].append(i)
        rng = np.random.default_rng(self.seed)
        sets = list(groups.values()); rng.shuffle(sets)
        n_test = int(self.test_size * len(scaffolds))
        test: List[int] = []
        for g in sets:
            if len(test) + len(g) <= n_test:
                test.extend(g)
        test = np.array(sorted(test))
        train = np.array(sorted(set(range(len(scaffolds))) - set(test)))
        return train, test

    def _random_holdout(self, y: np.ndarray):
        from sklearn.model_selection import train_test_split
        idx = np.arange(len(y))
        train, test = train_test_split(idx, test_size=self.test_size,
                                       stratify=y, random_state=self.seed)
        return np.array(sorted(train)), np.array(sorted(test))

    # -- scaffold-aware CV folds on the train pool ------------------------
    def _scaffold_folds(self, scaffolds: np.ndarray, train: np.ndarray) -> np.ndarray:
        groups = defaultdict(list)
        for pos, i in enumerate(train):
            groups[scaffolds[i]].append(pos)
        rng = np.random.default_rng(self.seed + 1)
        sets = list(groups.values()); rng.shuffle(sets)
        fold_of = np.full(len(train), -1, dtype=int)
        sizes = np.zeros(self.n_folds, dtype=int)
        for g in sorted(sets, key=len, reverse=True):
            f = int(np.argmin(sizes))          # greedy balance
            for pos in g:
                fold_of[pos] = f
            sizes[f] += len(g)
        return fold_of

    def split(self, scaffolds: np.ndarray, y: np.ndarray) -> Dict[str, np.ndarray]:
        if self.strategy == "scaffold":
            train, test = self._scaffold_holdout(scaffolds, y)
            fold_of = self._scaffold_folds(scaffolds, train)
        else:
            train, test = self._random_holdout(y)
            fold_of = np.full(len(train), -1)
            skf = StratifiedKFold(self.n_folds, shuffle=True, random_state=self.seed)
            for f, (_, va) in enumerate(skf.split(train, y[train])):
                fold_of[va] = f
        return {"train_idx": train, "test_idx": test, "fold_of_train": fold_of,
                "strategy": np.array(self.strategy)}


## `deepentxai/model.py`


In [ ]:
"""DEEPENTXAI — multimodal feature-fusion CNN-LSTM (Keras Functional API).

Four molecular views (ChemBERTa | Morgan | RDKit | MACCS) are encoded separately,
fused, reshaped into a sequence, then passed through Residual CNN blocks, Channel
Attention (squeeze-excite), a Bidirectional LSTM, Residual Dense blocks, Dropout
and fully-connected layers to a single Sigmoid activity neuron.

The CNN-LSTM identity of DEEPENTXAI is preserved and enhanced — not replaced.
"""
from __future__ import annotations

from typing import Dict

import keras
from keras import layers

# canonical input order used everywhere (model + training + prediction)
INPUT_ORDER = ["morgan", "rdkit", "maccs", "chemberta"]


# --------------------------------------------------------------------------- #
# Reusable blocks
# --------------------------------------------------------------------------- #
def _branch_encoder(inp, embed_dim: int, act: str, dropout: float, name: str):
    y = layers.Dense(256, name=f"{name}_dense1")(inp)
    y = layers.BatchNormalization(name=f"{name}_bn")(y)
    y = layers.Activation(act, name=f"{name}_act")(y)
    y = layers.Dropout(dropout, name=f"{name}_drop")(y)
    return layers.Dense(embed_dim, activation=act, name=f"{name}_embed")(y)


def _residual_cnn_block(x, filters: int, kernel: int, act: str, i: int):
    sc = x
    y = layers.Conv1D(filters, kernel, padding="same", name=f"cnn{i}_c1")(x)
    y = layers.BatchNormalization(name=f"cnn{i}_bn1")(y)
    y = layers.Activation(act, name=f"cnn{i}_a1")(y)
    y = layers.Conv1D(filters, kernel, padding="same", name=f"cnn{i}_c2")(y)
    y = layers.BatchNormalization(name=f"cnn{i}_bn2")(y)
    if sc.shape[-1] != filters:
        sc = layers.Conv1D(filters, 1, padding="same", name=f"cnn{i}_proj")(sc)
    y = layers.Add(name=f"cnn{i}_add")([sc, y])
    return layers.Activation(act, name=f"cnn{i}_out")(y)


def _channel_attention(x, ratio: int = 8, name: str = "se"):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling1D(name=f"{name}_gap")(x)
    s = layers.Dense(max(c // ratio, 4), activation="relu", name=f"{name}_fc1")(s)
    s = layers.Dense(c, activation="sigmoid", name=f"{name}_fc2")(s)
    s = layers.Reshape((1, c), name=f"{name}_rs")(s)
    return layers.Multiply(name=f"{name}_scale")([x, s])


def _residual_dense_block(x, units: int, act: str, dropout: float, i: int):
    sc = x
    y = layers.Dense(units, name=f"rd{i}_d1")(x)
    y = layers.BatchNormalization(name=f"rd{i}_bn1")(y)
    y = layers.Activation(act, name=f"rd{i}_a1")(y)
    y = layers.Dropout(dropout, name=f"rd{i}_drop")(y)
    y = layers.Dense(units, name=f"rd{i}_d2")(y)
    y = layers.BatchNormalization(name=f"rd{i}_bn2")(y)
    if sc.shape[-1] != units:
        sc = layers.Dense(units, name=f"rd{i}_proj")(sc)
    y = layers.Add(name=f"rd{i}_add")([sc, y])
    return layers.Activation(act, name=f"rd{i}_out")(y)


# --------------------------------------------------------------------------- #
# Full model
# --------------------------------------------------------------------------- #
def build_deepentxai(dims: Dict[str, int], hp: Dict) -> keras.Model:
    """Assemble the DEEPENTXAI multimodal fusion CNN-LSTM.

    dims : {'morgan':2048,'rdkit':100,'maccs':167,'chemberta':768}
    hp   : embed_dim, cnn_blocks, cnn_filters, cnn_kernel, lstm_units,
           dense_units, dropout, activation
    """
    e = int(hp["embed_dim"]); act = hp["activation"]; dr = float(hp["dropout"])
    seq_channels = 8

    inputs = {m: keras.Input((dims[m],), name=m) for m in INPUT_ORDER}
    encoded = [_branch_encoder(inputs[m], e, act, dr, m) for m in INPUT_ORDER]

    fused = layers.Concatenate(name="fusion")(encoded)          # (4*e,)
    fused = layers.BatchNormalization(name="fusion_bn")(fused)

    # project + reshape the fused vector into a (T, C) pseudo-sequence for CNN-LSTM
    x = layers.Dense(64 * seq_channels, activation=act, name="to_seq")(fused)
    x = layers.Reshape((64, seq_channels), name="reshape_seq")(x)

    for i in range(int(hp["cnn_blocks"])):
        x = _residual_cnn_block(x, int(hp["cnn_filters"]), int(hp["cnn_kernel"]), act, i)
    x = _channel_attention(x)
    x = layers.MaxPooling1D(2, name="pool")(x)
    x = layers.Bidirectional(layers.LSTM(int(hp["lstm_units"])), name="bilstm")(x)

    x = _residual_dense_block(x, int(hp["dense_units"]), act, dr, 0)
    x = layers.Dropout(dr, name="head_drop")(x)
    x = layers.Dense(int(hp["dense_units"]), activation=act, name="head_dense")(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32", name="activity")(x)

    return keras.Model(inputs=[inputs[m] for m in INPUT_ORDER], outputs=out, name="DEEPENTXAI")


## `deepentxai/train.py`


In [ ]:
"""Training utilities for DEEPENTXAI: data assembly, callbacks, class weights,
Optuna hyper-parameter optimisation, and fold/full training — all leakage-free."""
from __future__ import annotations

import os
from typing import Dict, List, Tuple

import numpy as np

from .model import INPUT_ORDER, build_deepentxai


# --------------------------------------------------------------------------- #
# Data assembly (leakage-free: ChemBERTa scaled on the train pool only)
# --------------------------------------------------------------------------- #
def load_matrices(cfg) -> Tuple[Dict[str, np.ndarray], np.ndarray, Dict[str, np.ndarray]]:
    import joblib
    feats = np.load(os.path.join(cfg.dir_features_raw, "features.npz"), allow_pickle=True)
    rsel = np.load(os.path.join(cfg.dir_features_selected, "rdkit_selected.npz"))
    split = dict(np.load(os.path.join(cfg.dir_splits, "split.npz"), allow_pickle=True))
    y = feats["y"].astype(int)
    tr = split["train_idx"]

    from sklearn.preprocessing import StandardScaler
    sb = StandardScaler().fit(feats["X_chemberta"][tr])            # fit on TRAIN pool only
    joblib.dump(sb, os.path.join(cfg.dir_features_selected, "chemberta_scaler.joblib"))

    X = {"morgan": feats["X_morgan"].astype("float32"),
         "rdkit": rsel["X_rdkit_selected"].astype("float32"),
         "maccs": feats["X_maccs"].astype("float32"),
         "chemberta": sb.transform(feats["X_chemberta"]).astype("float32")}
    return X, y, split


def slice_inputs(X: Dict[str, np.ndarray], idx: np.ndarray) -> List[np.ndarray]:
    # list in the model's canonical INPUT_ORDER (positional match, no name ambiguity)
    return [X[m][idx] for m in INPUT_ORDER]


def class_weights(y: np.ndarray) -> Dict[int, float]:
    n = len(y); npos = max(int(y.sum()), 1); nneg = max(int((y == 0).sum()), 1)
    return {0: n / (2 * nneg), 1: n / (2 * npos)}


# --------------------------------------------------------------------------- #
# Compile + train one model
# --------------------------------------------------------------------------- #
def _compile(model, hp, cfg):
    import keras
    opt_name = hp.get("optimizer", "adam")
    lr = float(hp.get("learning_rate", cfg["train"]["learning_rate"]))
    clip = float(cfg["train"].get("gradient_clipnorm", 1.0))
    opt = {"adam": keras.optimizers.Adam, "nadam": keras.optimizers.Nadam,
           "rmsprop": keras.optimizers.RMSprop}.get(opt_name, keras.optimizers.Adam)(
        learning_rate=lr, clipnorm=clip)
    model.compile(optimizer=opt, loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="auc")])
    return model


def train_model(X, y, dims, hp, cfg, tr_idx, va_idx, epochs, patience,
                ckpt_path=None, verbose=0):
    import keras
    model = _compile(build_deepentxai(dims, hp), hp, cfg)
    cbs: List = [keras.callbacks.EarlyStopping("val_loss", patience=patience,
                                               restore_best_weights=True)]
    cbs.append(keras.callbacks.ReduceLROnPlateau("val_loss",
               patience=int(cfg["train"]["reduce_lr_patience"]), factor=0.5, min_lr=1e-6))
    if ckpt_path:
        cbs.append(keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_auc",
                   mode="max", save_best_only=True))
    cw = class_weights(y[tr_idx]) if cfg["train"].get("class_weight", True) else None
    hist = model.fit(slice_inputs(X, tr_idx), y[tr_idx],
                     validation_data=(slice_inputs(X, va_idx), y[va_idx]),
                     epochs=epochs, batch_size=int(hp.get("batch_size", cfg["train"]["batch_size"])),
                     class_weight=cw, callbacks=cbs, verbose=verbose)
    return model, hist.history


# --------------------------------------------------------------------------- #
# Optuna hyper-parameter search (leakage-free: only the train pool folds used)
# --------------------------------------------------------------------------- #
class OptunaTuner:
    def __init__(self, cfg, X, y, dims, split, log):
        self.cfg, self.X, self.y, self.dims, self.split, self.log = cfg, X, y, dims, split, log
        tr = split["train_idx"]; fold = split["fold_of_train"]
        self.pool = tr
        self.va = tr[fold == 0]                     # validation fold
        self.tr = tr[fold != 0]                     # training folds

    def _space(self, trial) -> dict:
        return {
            "embed_dim": trial.suggest_categorical("embed_dim", [96, 128, 192]),
            "cnn_blocks": trial.suggest_int("cnn_blocks", 1, 3),
            "cnn_filters": trial.suggest_categorical("cnn_filters", [32, 64, 128]),
            "cnn_kernel": trial.suggest_categorical("cnn_kernel", [3, 5]),
            "lstm_units": trial.suggest_categorical("lstm_units", [64, 128]),
            "dense_units": trial.suggest_categorical("dense_units", [64, 128, 256]),
            "dropout": trial.suggest_float("dropout", 0.2, 0.5),
            "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
            "optimizer": trial.suggest_categorical("optimizer", ["adam", "nadam"]),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        }

    def objective(self, trial):
        from sklearn.metrics import roc_auc_score
        hp = self._space(trial)
        model, _ = train_model(self.X, self.y, self.dims, hp, self.cfg,
                               self.tr, self.va, epochs=int(self.cfg["train"]["hpo_epochs"]),
                               patience=5, verbose=0)
        p = model.predict(slice_inputs(self.X, self.va), verbose=0).ravel()
        import keras; keras.backend.clear_session()
        return roc_auc_score(self.y[self.va], p)

    def run(self):
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=self.cfg.seed))
        study.optimize(self.objective, n_trials=int(self.cfg["optuna"]["n_trials"]),
                       timeout=int(self.cfg["optuna"]["timeout_min"]) * 60,
                       show_progress_bar=False)
        self.log.info(f"best val ROC-AUC={study.best_value:.4f}  params={study.best_params}")
        return study


## `deepentxai/evaluate.py`


In [ ]:
"""Rigorous binary-classification evaluation: 13 metrics + publication figures."""
from __future__ import annotations

import os
from typing import Dict

import numpy as np
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, brier_score_loss,
                             cohen_kappa_score, confusion_matrix, f1_score,
                             log_loss, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    p = np.clip(y_prob, 1e-7, 1 - 1e-7)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "log_loss": log_loss(y_true, p),
        "brier_score": brier_score_loss(y_true, y_prob),
    }


def _style():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"savefig.dpi": 300, "figure.dpi": 120, "font.size": 12,
                         "axes.grid": True, "grid.alpha": 0.3})
    return plt


def plot_curves(y_true: np.ndarray, y_prob: np.ndarray, out_dir: str, history=None) -> None:
    """ROC, PR, confusion matrix, calibration and (optional) learning curves."""
    from sklearn.metrics import (roc_curve, precision_recall_curve,
                                 ConfusionMatrixDisplay, average_precision_score, roc_auc_score)
    from sklearn.calibration import calibration_curve
    plt = _style()
    os.makedirs(out_dir, exist_ok=True)
    y_pred = (y_prob >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(y_true, y_prob):.3f}")
    ax.plot([0, 1], [0, 1], "--", c="grey"); ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate"); ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "F02_roc_curve.png")); plt.close(fig)

    pr, rc, _ = precision_recall_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(rc, pr, lw=2, label=f"PR-AUC = {average_precision_score(y_true, y_prob):.3f}")
    ax.axhline(y_true.mean(), ls="--", c="grey"); ax.set_xlabel("Recall")
    ax.set_ylabel("Precision"); ax.legend(loc="lower left")
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "F03_pr_curve.png")); plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]),
                           display_labels=["Inactive", "Active"]).plot(ax=ax, cmap="Blues", colorbar=False)
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "F05_confusion_matrix.png")); plt.close(fig)

    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(mean_pred, frac_pos, "o-", label="DEEPENTXAI")
    ax.plot([0, 1], [0, 1], "--", c="grey", label="perfectly calibrated")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed frequency"); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "F04_calibration_curve.png")); plt.close(fig)

    if history is not None:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        for k in ("loss", "val_loss"):
            if k in history: ax[0].plot(history[k], label=k)
        ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
        for k in history:
            if "auc" in k.lower(): ax[1].plot(history[k], label=k)
        ax[1].set_xlabel("epoch"); ax[1].set_ylabel("AUC"); ax[1].legend()
        fig.tight_layout(); fig.savefig(os.path.join(out_dir, "F06_learning_curve.png")); plt.close(fig)


## `deepentxai/explain.py`


In [ ]:
"""Multi-method explainability for DEEPENTXAI.

  * Permutation importance  — modality-level and RDKit-descriptor-level (robust).
  * Integrated Gradients    — per-descriptor attribution on the RDKit branch (TF).
  * SHAP                    — GradientExplainer over the RDKit branch (guarded).
  * LIME                    — instance-level tabular explanations (guarded).

The RDKit descriptor block is the human-interpretable channel, so global feature
explanations target it; the other modalities are summarised at the block level.
"""
from __future__ import annotations

from typing import Dict, List

import numpy as np
from sklearn.metrics import roc_auc_score

# canonical input order (must match the model)
_ORDER = ["morgan", "rdkit", "maccs", "chemberta"]
_RDKIT = 1


def _predict(model, X: List[np.ndarray]) -> np.ndarray:
    return model.predict(X, verbose=0).ravel()


# --------------------------------------------------------------------------- #
# Permutation importance
# --------------------------------------------------------------------------- #
def permutation_modality(model, X: List[np.ndarray], y: np.ndarray,
                         n_repeats: int = 5, seed: int = 42) -> Dict[str, float]:
    rng = np.random.default_rng(seed)
    base = roc_auc_score(y, _predict(model, X))
    out = {}
    for m in range(len(X)):
        drops = []
        for _ in range(n_repeats):
            Xp = [x.copy() for x in X]
            Xp[m] = Xp[m][rng.permutation(len(y))]
            drops.append(base - roc_auc_score(y, _predict(model, Xp)))
        out[_ORDER[m]] = float(np.mean(drops))
    return out


def permutation_descriptor(model, X: List[np.ndarray], y: np.ndarray,
                           names: List[str], n_repeats: int = 3, seed: int = 42):
    import pandas as pd
    rng = np.random.default_rng(seed)
    base = roc_auc_score(y, _predict(model, X))
    imp = np.zeros(X[_RDKIT].shape[1])
    for j in range(X[_RDKIT].shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = [x.copy() for x in X]
            col = Xp[_RDKIT][:, j].copy()
            Xp[_RDKIT][:, j] = col[rng.permutation(len(y))]
            drops.append(base - roc_auc_score(y, _predict(model, Xp)))
        imp[j] = np.mean(drops)
    return (pd.DataFrame({"descriptor": names, "importance": imp})
            .sort_values("importance", ascending=False).reset_index(drop=True))


# --------------------------------------------------------------------------- #
# Integrated Gradients (RDKit branch)
# --------------------------------------------------------------------------- #
def integrated_gradients_rdkit(model, X: List[np.ndarray], names: List[str],
                               steps: int = 32, n_samples: int = 300, seed: int = 42):
    import pandas as pd
    import tensorflow as tf
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X[0]), size=min(n_samples, len(X[0])), replace=False)
    Xs = [tf.convert_to_tensor(x[idx]) for x in X]
    baseline = [tf.zeros_like(t) for t in Xs]
    total = tf.zeros_like(Xs[_RDKIT])
    for a in np.linspace(0, 1, steps):
        interp = [baseline[k] + a * (Xs[k] - baseline[k]) for k in range(len(Xs))]
        with tf.GradientTape() as tape:
            tape.watch(interp[_RDKIT])
            out = model(interp, training=False)
        g = tape.gradient(out, interp[_RDKIT])
        total += g
    ig = (Xs[_RDKIT] - baseline[_RDKIT]) * (total / steps)
    attr = tf.reduce_mean(tf.abs(ig), axis=0).numpy()
    return (pd.DataFrame({"descriptor": names, "attribution": attr})
            .sort_values("attribution", ascending=False).reset_index(drop=True))


# --------------------------------------------------------------------------- #
# SHAP (guarded) — GradientExplainer over the RDKit branch
# --------------------------------------------------------------------------- #
def shap_rdkit(model, X: List[np.ndarray], names: List[str], out_dir: str,
               n_bg: int = 100, n_explain: int = 200, seed: int = 42, log=None):
    try:
        import shap, matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        rng = np.random.default_rng(seed)
        bg = [x[rng.choice(len(x), n_bg, replace=False)] for x in X]
        ex = [x[rng.choice(len(x), n_explain, replace=False)] for x in X]
        sv = shap.GradientExplainer(model, bg).shap_values(ex)
        sv_rdkit = sv[_RDKIT] if isinstance(sv, list) else sv
        sv_rdkit = np.array(sv_rdkit).reshape(len(ex[_RDKIT]), -1)
        shap.summary_plot(sv_rdkit, ex[_RDKIT], feature_names=names, show=False, max_display=20)
        plt.tight_layout(); plt.savefig(f"{out_dir}/shap_summary.png", dpi=300, bbox_inches="tight"); plt.close()
        return True
    except Exception as e:
        if log: log.info(f"SHAP skipped: {type(e).__name__}: {e}")
        return False


# --------------------------------------------------------------------------- #
# LIME (guarded) — a few representative instances
# --------------------------------------------------------------------------- #
def lime_instances(model, X: List[np.ndarray], names: List[str], out_dir: str,
                   instances: List[int], log=None):
    try:
        from lime.lime_tabular import LimeTabularExplainer
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        rdk = X[_RDKIT]

        def predict_fn(z):                       # vary RDKit, hold others at their row values
            n = len(z); reps = []
            for k in range(len(X)):
                reps.append(np.repeat(X[k][:1], n, axis=0) if k != _RDKIT else z)
            p = model.predict(reps, verbose=0).ravel()
            return np.column_stack([1 - p, p])

        expl = LimeTabularExplainer(rdk, feature_names=names,
                                    class_names=["Inactive", "Active"], mode="classification")
        for i in instances:
            e = expl.explain_instance(rdk[i], predict_fn, num_features=10)
            fig = e.as_pyplot_figure(); fig.tight_layout()
            fig.savefig(f"{out_dir}/lime_instance_{i}.png", dpi=200, bbox_inches="tight"); plt.close(fig)
        return True
    except Exception as e:
        if log: log.info(f"LIME skipped: {type(e).__name__}: {e}")
        return False


## `deepentxai/scoring.py`


In [ ]:
"""DEEPENTXAI compound scoring & ranking.

Turns raw activity probabilities into a 0-100 **DEEPENTXAI Score** that blends the
predicted probability of activity with prediction confidence (distance from the
decision boundary), then ranks compounds for downstream drug-discovery triage.
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def deepentxai_score(prob: np.ndarray, confidence_weight: float = 0.3) -> np.ndarray:
    """Score in [0, 100]: mostly P(active), boosted by boundary confidence.

    score = 100 * ( (1-w) * p  +  w * p * |2p-1| )
    where |2p-1| is 0 at the 0.5 boundary and 1 at the extremes.
    """
    p = np.clip(prob, 0.0, 1.0)
    conf = np.abs(2 * p - 1.0)
    return 100.0 * ((1 - confidence_weight) * p + confidence_weight * p * conf)


def rank_compounds(smiles: np.ndarray, prob: np.ndarray, y_true: np.ndarray | None = None,
                   confidence_weight: float = 0.3) -> pd.DataFrame:
    """Ranked table: SMILES, P(active), DEEPENTXAI Score, predicted call, (truth)."""
    score = deepentxai_score(prob, confidence_weight)
    df = pd.DataFrame({
        "smiles": smiles,
        "prob_active": np.round(prob, 4),
        "deepentxai_score": np.round(score, 2),
        "prediction": np.where(prob >= 0.5, "Active", "Inactive"),
    })
    if y_true is not None:
        df["observed"] = np.where(np.asarray(y_true) == 1, "Active", "Inactive")
    return df.sort_values("deepentxai_score", ascending=False).reset_index(drop=True)


# =============================================================================
# Retrospective virtual-screening validation of the ranking.
#
# A ranking is only scientifically meaningful if the actives really do pile up at
# the top. On the scaffold-disjoint hold-out (labels known) we quantify that with
# the standard early-recognition metrics used in virtual screening:
#   * Enrichment Factor (EF@x%)  = (hit-rate in the top x%) / (overall active rate).
#                                  EF = 1 means "no better than random"; higher is
#                                  better; the ceiling is 1/active_rate.
#   * hit-rate@x%                = fraction of the top x% that are truly active.
#   * precision@K                = hit-rate among the top-K compounds.
#   * accumulation curve         = actives recovered vs fraction screened (its area
#                                  equals the ranking ROC-AUC).
# All of these are computed on TRUE labels, so they are directly falsifiable.
# =============================================================================
def _ranked_labels(y_true: np.ndarray, score: np.ndarray) -> np.ndarray:
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")
    return np.asarray(y_true, dtype=int)[order]


def enrichment(y_true: np.ndarray, score: np.ndarray,
               fractions=(0.01, 0.05, 0.10, 0.20)) -> dict:
    y = _ranked_labels(y_true, score)
    N, n_act = len(y), int(y.sum())
    base = n_act / N if N else float("nan")               # random active rate
    out = {"n_total": int(N), "n_active": int(n_act),
           "active_rate": round(base, 4), "max_EF": round(1.0 / base, 2) if base else float("nan")}
    rows = {}
    for f in fractions:
        k = max(1, int(round(f * N)))
        hits = int(y[:k].sum())
        hit_rate = hits / k
        rows[f"top_{int(f*100)}pct"] = {
            "k": k, "hits": hits,
            "hit_rate": round(hit_rate, 4),
            "EF": round(hit_rate / base, 2) if base else float("nan"),
        }
    out["by_fraction"] = rows
    return out


def precision_at_k(y_true: np.ndarray, score: np.ndarray,
                   ks=(10, 25, 50, 100, 250, 500)) -> dict:
    y = _ranked_labels(y_true, score)
    N = len(y)
    return {f"P@{k}": round(float(y[:min(k, N)].sum()) / min(k, N), 4) for k in ks}


def accumulation_curve(y_true: np.ndarray, score: np.ndarray):
    """Return (fraction_screened, fraction_of_actives_recovered) for plotting."""
    y = _ranked_labels(y_true, score)
    N, n_act = len(y), int(y.sum())
    frac_screened = np.arange(1, N + 1) / N
    frac_found = np.cumsum(y) / (n_act if n_act else 1)
    return frac_screened, frac_found


## `deepentxai/predict.py`


In [ ]:
"""End-to-end prediction pipeline: raw SMILES -> DEEPENTXAI activity + score.

Loads the persisted artifacts (model, RDKit selector, ChemBERTa scaler) and applies
the exact same standardisation and featurisation used in training, so predictions
on new compounds are fully reproducible.
"""
from __future__ import annotations

import os
from typing import List

import joblib
import numpy as np
import pandas as pd
from rdkit import Chem

from .features import FeatureGenerator
from .model import INPUT_ORDER
from .scoring import rank_compounds
from .standardize import MoleculeStandardizer


class PredictionPipeline:
    def __init__(self, cfg):
        import keras
        self.cfg = cfg
        self.std = MoleculeStandardizer(cfg["standardize"])
        self.fg = FeatureGenerator(cfg["features"])
        self.rdkit_selector = joblib.load(os.path.join(cfg.dir_features_selected, "rdkit_selector.joblib"))
        self.chemberta_scaler = joblib.load(os.path.join(cfg.dir_features_selected, "chemberta_scaler.joblib"))
        self.model = keras.models.load_model(os.path.join(cfg.dir_models, "deepentxai_best.keras"))

    def _featurize(self, smiles: List[str]):
        std_smiles, mols = [], []
        for s in smiles:
            canon, _ = self.std(s)
            if canon is None:
                continue
            m = Chem.MolFromSmiles(canon)
            if m is not None:
                std_smiles.append(canon); mols.append(m)
        morgan = np.stack([self.fg.morgan(m) for m in mols]).astype("float32")
        maccs = np.stack([self.fg.maccs(m) for m in mols]).astype("float32")
        rdkit = np.array([self.fg.rdkit(m) for m in mols], dtype=np.float64)
        rdkit = np.where(np.isfinite(rdkit), rdkit, np.nan).astype("float32")
        rdkit_sel = self.rdkit_selector.transform(rdkit)
        chemberta = self.chemberta_scaler.transform(self.fg.chemberta(std_smiles)).astype("float32")
        X = {"morgan": morgan, "rdkit": rdkit_sel, "maccs": maccs, "chemberta": chemberta}
        return std_smiles, [X[m] for m in INPUT_ORDER]

    def predict(self, smiles: List[str]) -> pd.DataFrame:
        std_smiles, X = self._featurize(smiles)
        prob = self.model.predict(X, verbose=0).ravel()
        return rank_compounds(np.array(std_smiles), prob)


## `deepentxai/__init__.py`


# 2 · Pipeline stages (numbered)
The orchestrator plus stages 01–15. Core stages 01–08 build the model and its figures; extras 09–15 add baselines, regression, the leakage demonstration and the reviewer-driven robustness analyses.

## `00_run_all.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 00 | ORCHESTRATOR  — one command runs the entire pipeline
=================================================================================
From raw public data to the final figures, in order, with one process per stage
(each stage gets a clean TensorFlow/Keras state):

    01  download + preprocess ...... ChEMBL + PubChem -> standardise -> label
                                     (activity-gap high-confidence subset)
    02  feature engineering ........ Morgan+RDKit+MACCS+ChemBERTa, scaffold split,
                                     train-only RDKit descriptor selection
    03  train + validate ........... Optuna HPO -> 5-fold scaffold CV -> hold-out
    04  ensemble + calibrate ....... 5-fold CNN-LSTM ensemble (the headline model),
                                     isotonic calibration + OOF threshold, saves
                                     the 5 fold weights + OOF/test predictions
    05  conformal selective ........ honest high-accuracy statement WITH coverage
    06  explainability ............. permutation / Integrated Gradients / SHAP / LIME
    07  scoring + ranking .......... DEEPENTXAI score + hit list, validated by
                                     enrichment factor / precision@K on the hold-out
    08  reports + figures .......... publication metrics table + summary figure

Everything is scaffold-disjoint (leakage-free). Data lands under 02_Data/,
results under 03_Results/ — all numbered.

USAGE
    P=~/miniconda3/envs/ENT/bin/python
    $P 00_run_all.py                      # run every stage 01..08
    $P 00_run_all.py --only 04 05         # run just these stages
    $P 00_run_all.py --from 03            # resume from stage 03 onward
    $P 00_run_all.py --to 02              # only 01..02
    $P 00_run_all.py --threads 8          # CPU threads per stage (default 8)
    $P 00_run_all.py --list               # show the stage table and exit

Notes
  * No GPU required — tuned for CPU (DEEPENT_THREADS default 8).
  * Stage 01 reuses cached raw downloads if 02_Data/01_Raw is already populated
    (config download.reuse_cache: true), so re-runs skip the network.
  * Stages write into fixed numbered dirs; re-running a stage overwrites cleanly.
=================================================================================
"""
from __future__ import annotations

import argparse
import os
import subprocess
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))

# (stage id, script filename, one-line description)
STAGES = [
    ("01", "01_download_and_preprocess.py",       "download ChEMBL+PubChem, standardise, activity-gap labelling"),
    ("02", "02_feature_engineering.py",           "Morgan+RDKit+MACCS+ChemBERTa, scaffold split, descriptor selection"),
    ("03", "03_train_and_validate.py",            "Optuna HPO, 5-fold scaffold CV, hold-out test"),
    ("04", "04_ensemble_calibrate_threshold.py",  "5-fold CNN-LSTM ensemble + calibration + fold weights"),
    ("05", "05_conformal_selective.py",           "conformal selective prediction (accuracy vs coverage)"),
    ("06", "06_explainability.py",                "permutation / IntegratedGradients / SHAP / LIME"),
    ("07", "07_compound_scoring_and_ranking.py",  "DEEPENTXAI score + ranking, validated by enrichment (EF/precision@K)"),
    ("08", "08_reports_and_figures.py",           "publication metrics table + summary figure"),
]
IDS = [s[0] for s in STAGES]

# Optional secondary analyses — NOT part of the default 01..08 run.
# Invoke explicitly, e.g.  00_run_all.py --only 09 10 11
EXTRAS = [
    ("09", "09_baselines_and_stacking.py",  "XGBoost/RandomForest baselines + CNN+XGB+RF stacking"),
    ("10", "10_regression_ecoli_mic.py",    "E. coli MIC pMIC regression (secondary framing)"),
    ("11", "11_leakage_demonstration.py",   "leakage control: random-split artifact (integrity check)"),
    ("12", "12_assay_source_bias.py",       "assay/source-bias analysis (reviewer R2.6)"),
    ("13", "13_imbalanced_evaluation.py",   "highly-imbalanced stress test, PR-AUC/EF (reviewer R1.2)"),
    ("14", "14_feature_stability.py",       "bootstrap + across-fold feature-importance stability (R2.9)"),
    ("15", "15_external_cross_source.py",   "external validation: train ChEMBL / test PubChem (R2.3)"),
]
ALL = STAGES + EXTRAS


def select(args) -> list:
    ids = IDS
    if args.only:                                   # --only may name core OR extra stages
        want = set(args.only)
        return [s for s in ALL if s[0] in want]
    if args.from_:
        ids = IDS[IDS.index(args.from_):]
    if args.to:
        ids = [i for i in ids if i <= args.to]
    return [s for s in STAGES if s[0] in set(ids)]   # default run = core 01..08 only


def main() -> int:
    ap = argparse.ArgumentParser(description="DEEPENTXAI_FINAL orchestrator")
    ap.add_argument("--only", nargs="+", metavar="ID", help="run only these stage ids, e.g. 04 05")
    ap.add_argument("--from", dest="from_", metavar="ID", help="resume from this stage id onward")
    ap.add_argument("--to", metavar="ID", help="stop after this stage id")
    ap.add_argument("--threads", type=int, default=int(os.environ.get("DEEPENT_THREADS", "8")),
                    help="CPU threads per stage (default 8)")
    ap.add_argument("--list", action="store_true", help="print the stage table and exit")
    args = ap.parse_args()

    if args.list:
        print("\nDEEPENTXAI_FINAL core pipeline (default run = 01..08):")
        for sid, script, desc in STAGES:
            print(f"  {sid}  {script:38s} {desc}")
        print("\nOptional secondary analyses (run with --only):")
        for sid, script, desc in EXTRAS:
            print(f"  {sid}  {script:38s} {desc}")
        return 0

    stages = select(args)
    if not stages:
        print("no stages selected"); return 1

    env = dict(os.environ)
    env["DEEPENT_THREADS"] = str(args.threads)
    for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
              "NUMEXPR_NUM_THREADS", "TF_NUM_INTRAOP_THREADS"):
        env[v] = str(args.threads)
    env.setdefault("DEEPENT_CONFIG", os.path.join(HERE, "config.yaml"))

    py = sys.executable
    print("=" * 79)
    print(f"DEEPENTXAI_FINAL orchestrator | python={py} | threads={args.threads}")
    print(f"stages: {', '.join(s[0] for s in stages)}")
    print("=" * 79)

    t0 = time.time()
    for sid, script, desc in stages:
        print(f"\n>>> STAGE {sid}  {desc}\n    {script}", flush=True)
        st = time.time()
        r = subprocess.run([py, os.path.join(HERE, script)], env=env, cwd=HERE)
        dt = time.time() - st
        if r.returncode != 0:
            print(f"\n!!! STAGE {sid} FAILED (exit {r.returncode}) after {dt/60:.1f} min. Stopping.")
            return r.returncode
        print(f"<<< STAGE {sid} done in {dt/60:.1f} min", flush=True)

    print(f"\n{'='*79}\nALL SELECTED STAGES COMPLETE in {(time.time()-t0)/60:.1f} min")
    print("results: 03_Results/  (01_Figures, 02_Metrics, 03_Model, 04_Explainability, ...)")
    print("=" * 79)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `01_download_and_preprocess.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 01 | Data Download & Preprocessing
=================================================================================
Automated, reproducible acquisition of experimentally validated bioactivity data
from ChEMBL and PubChem BioAssay, followed by professional cheminformatics
standardisation, configurable binary labelling, conflict resolution and
de-duplication. Produces a clean modelling dataset + a preprocessing report.

Inputs  : 01_Code/config.yaml  (+ cached 02_Data/01_Raw/*.csv, else the public APIs)
Outputs : 02_Data/02_Processed/labelled.csv
          03_Results/02_Metrics/nb1_preprocessing_report.json
          03_Results/01_Figures/01_dataset_overview.png
          03_Results/06_Logs/NB1.log

Run     : ~/miniconda3/envs/ENT/bin/python 01_data_download_and_preprocessing.py
=================================================================================
"""
from __future__ import annotations

import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))          # find the deepentxai package
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, detect_gpu, save_json
from deepentxai.download import DataDownloader
from deepentxai.standardize import MoleculeStandardizer
from deepentxai.labeling import ActivityLabeler


def main() -> None:
    # --- 1. setup -------------------------------------------------------------
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB1", cfg.dir_logs)
    report: dict = {}
    log.info(f"DEEPENTXAI-01 | seed={cfg.seed} | GPU={detect_gpu()} | "
             f"threshold={cfg['labeling']['affinity_threshold_uM']} uM")
    log.info(f"Target: {cfg['target']['description']}")

    # --- 2. download (cached if already fetched) ------------------------------
    data = DataDownloader(cfg).download()
    chembl = data.get("ChEMBL", pd.DataFrame())
    pubchem = data.get("PubChem", pd.DataFrame())
    report["download"] = {"chembl_rows": int(len(chembl)), "pubchem_rows": int(len(pubchem))}
    log.info(f"downloaded  ChEMBL={chembl.shape}  PubChem={pubchem.shape}")

    # --- 3. standardise structures -------------------------------------------
    std = MoleculeStandardizer(cfg["standardize"])
    all_smiles = pd.unique(pd.concat([
        chembl.get("smiles", pd.Series(dtype=str)),
        pubchem.get("smiles", pd.Series(dtype=str))], ignore_index=True))
    canon_map, mw_map = {}, {}
    for s in all_smiles:
        canon_map[s], mw_map[s] = std(s)
    report["standardization"] = std.report()
    log.info(f"standardisation: {report['standardization']}")

    # --- 4. label -------------------------------------------------------------
    labeler = ActivityLabeler(cfg["labeling"])
    rows = []
    for r in chembl.itertuples(index=False):
        canon = canon_map.get(r.smiles)
        if canon is None:
            continue
        lab = labeler.label_one(r.type, r.value, r.units, mw_map.get(r.smiles))
        if lab is not None:
            rows.append({"smiles": canon, "lab": lab, "source": "ChEMBL"})
    for r in pubchem.itertuples(index=False):
        canon = canon_map.get(r.smiles)
        if canon is not None:
            rows.append({"smiles": canon, "lab": int(r.label), "source": "PubChem"})
    measurements = pd.DataFrame(rows)
    report["labelling"] = {"labelled_measurements": int(len(measurements)),
                           "active_frac": round(float(measurements["lab"].mean()), 4)}
    log.info(f"labelled measurements: {report['labelling']}")

    # --- 5. resolve conflicts + de-duplicate ---------------------------------
    final = labeler.resolve(measurements, "smiles", "lab")
    srcmix = measurements.groupby("smiles")["source"].agg(lambda s: "+".join(sorted(set(s))))
    final["sources"] = final["smiles"].map(srcmix)
    report["final"] = {"unique_compounds": int(len(final)),
                       "active": int(final["label"].sum()),
                       "inactive": int((final["label"] == 0).sum()),
                       "active_frac": round(float(final["label"].mean()), 4)}
    log.info(f"final dataset: {report['final']}")

    # --- 6. save -------------------------------------------------------------
    out_csv = os.path.join(cfg.dir_processed, "labelled.csv")
    final.to_csv(out_csv, index=False)
    save_json(report, os.path.join(cfg.dir_reports, "nb1_preprocessing_report.json"))
    log.info(f"saved {len(final)} unique compounds -> {out_csv}")

    # --- 7. QC figure --------------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    final["label"].map({1: "Active", 0: "Inactive"}).value_counts().plot.bar(
        ax=ax[0], color=["#2a9d8f", "#e76f51"], edgecolor="black")
    ax[0].set_ylabel("compounds"); ax[0].set_xlabel("")
    final["sources"].value_counts().plot.bar(ax=ax[1], color="#4c72b0", edgecolor="black")
    ax[1].set_xlabel("")
    fig.tight_layout()
    figp = os.path.join(cfg.dir_figures, "F01_dataset_overview.png")
    fig.savefig(figp, dpi=150, bbox_inches="tight"); plt.close(fig)

    print("\n================ DEEPENTXAI-01 complete ================")
    print(f"  unique compounds : {report['final']['unique_compounds']}")
    print(f"  active / inactive: {report['final']['active']} / {report['final']['inactive']}")
    print(f"  active fraction  : {report['final']['active_frac']}")
    print(f"  dataset          : {out_csv}")
    print(f"  QC figure        : {figp}")


if __name__ == "__main__":
    main()


## `02_feature_engineering.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 02 | Feature Engineering
=================================================================================
Generate four complementary molecular representations for every compound:
    Morgan ECFP4 (2048) | RDKit descriptors (~210) | MACCS (167) | ChemBERTa (768)
Create a leakage-free split (scaffold-disjoint hold-out + Stratified K-Fold on the
train pool), then fit numerical-descriptor selection (variance -> correlation ->
Mutual Information -> RFE) on the TRAIN pool only and persist it for prediction.

Inputs  : 02_Data/02_Processed/labelled.csv
Outputs : 02_Data/03_Features_Raw/features.npz      (all raw representations + scaffold)
          02_Data/05_Splits/split.npz               (train/test/fold indices)
          02_Data/04_Features_Selected/rdkit_selector.joblib
          02_Data/04_Features_Selected/selected_rdkit_features.json
          02_Data/04_Features_Selected/rdkit_selected.npz
          03_Results/02_Metrics/nb2_feature_report.json
Run     : ~/miniconda3/envs/ENT/bin/python 02_feature_engineering.py
=================================================================================
"""
from __future__ import annotations

import os
import sys

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.features import FeatureGenerator, RDKIT_NAMES
from deepentxai.splits import DataSplitter
from deepentxai.selection import FeatureSelector


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB2", cfg.dir_logs)
    report: dict = {}

    df = pd.read_csv(os.path.join(cfg.dir_processed, "labelled.csv"))
    smiles = df["smiles"].tolist()
    y = df["label"].astype(int).to_numpy()
    log.info(f"loaded {len(df)} compounds  (active {y.mean():.3f})")

    # --- 1. generate all representations -------------------------------------
    fg = FeatureGenerator(cfg["features"])
    feats, keep = fg.generate(smiles, log=log)
    y = y[keep]; smiles = [smiles[i] for i in keep]
    scaffold = feats["scaffold"]
    np.savez_compressed(os.path.join(cfg.dir_features_raw, "features.npz"),
                        y=y, smiles=np.array(smiles), rdkit_cols=np.array(RDKIT_NAMES), **feats)
    report["features"] = {k: list(v.shape) for k, v in feats.items() if k.startswith("X_")}
    log.info(f"features: {report['features']}")

    # --- 2. leakage-free split -----------------------------------------------
    split = DataSplitter(cfg, seed=cfg.seed).split(scaffold, y)
    np.savez_compressed(os.path.join(cfg.dir_splits, "split.npz"), **split)
    tr, te = split["train_idx"], split["test_idx"]
    # verify scaffold disjointness of the hold-out
    disjoint = len(set(scaffold[tr]) & set(scaffold[te])) == 0
    report["split"] = {"strategy": str(split["strategy"]), "n_train": int(len(tr)),
                       "n_test": int(len(te)), "n_folds": int(cfg["split"]["n_folds"]),
                       "holdout_scaffold_disjoint": bool(disjoint)}
    log.info(f"split: {report['split']}")
    assert disjoint or cfg["split"]["strategy"] != "scaffold", "hold-out not scaffold-disjoint"

    # --- 3. RDKit descriptor selection (fit on TRAIN pool only) ---------------
    sel = FeatureSelector(cfg, feature_names=list(RDKIT_NAMES), seed=cfg.seed)
    sel.fit(feats["X_rdkit"][tr], y[tr])
    X_rdkit_sel = sel.transform(feats["X_rdkit"])          # applied to ALL rows (fit on train)
    joblib.dump(sel, os.path.join(cfg.dir_features_selected, "rdkit_selector.joblib"))
    save_json({"selected_rdkit_features": sel.selected_names_,
               "n_selected": len(sel.selected_names_)},
              os.path.join(cfg.dir_features_selected, "selected_rdkit_features.json"))
    np.savez_compressed(os.path.join(cfg.dir_features_selected, "rdkit_selected.npz"),
                        X_rdkit_selected=X_rdkit_sel)
    report["selection"] = {"n_selected_rdkit": len(sel.selected_names_),
                           "from_total_rdkit": len(RDKIT_NAMES)}
    log.info(f"selection: {report['selection']}")

    save_json(report, os.path.join(cfg.dir_reports, "nb2_feature_report.json"))
    print("\n================ DEEPENTXAI-02 complete ================")
    print(f"  molecules       : {len(y)}")
    print(f"  Morgan/RDKit/MACCS/ChemBERTa : {feats['X_morgan'].shape[1]}/"
          f"{feats['X_rdkit'].shape[1]}/{feats['X_maccs'].shape[1]}/{feats['X_chemberta'].shape[1]}")
    print(f"  selected RDKit  : {len(sel.selected_names_)} / {len(RDKIT_NAMES)}")
    print(f"  split           : {report['split']['n_train']} train / {report['split']['n_test']} test "
          f"(scaffold-disjoint={report['split']['holdout_scaffold_disjoint']})")


if __name__ == "__main__":
    main()


## `03_train_and_validate.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 04 | Hyperparameter Optimization, Training & Validation
=================================================================================
1. Optuna search over model + training hyper-parameters (validated on a held-out
   scaffold fold of the train pool — no test leakage).
2. Stratified/scaffold 5-fold cross-validation with the best configuration.
3. Retrain the best configuration on the full train pool and evaluate ONCE on the
   independent scaffold-disjoint hold-out test set (13 metrics + publication curves).

Inputs  : 02_Data/03_Features_Raw + 04_Features_Selected + 05_Splits
Outputs : 03_Results/03_Model/deepentxai_best.keras (+ best_hparams.json)
          03_Results/07_Optuna/study.pkl
          03_Results/02_Metrics/{cv_metrics.json, test_metrics.json}
          03_Results/01_Figures/{roc,pr,confusion,calibration,learning}_curve.png
Run     : ~/miniconda3/envs/ENT/bin/python 04_train_and_validate.py
=================================================================================
"""
from __future__ import annotations

import os
import sys

import joblib
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, detect_gpu, save_json
from deepentxai.train import (load_matrices, slice_inputs, train_model, OptunaTuner)
from deepentxai.evaluate import compute_metrics, plot_curves
from deepentxai.model import INPUT_ORDER


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB4", cfg.dir_logs)
    gpu = detect_gpu()
    if cfg["train"].get("mixed_precision") == "auto" and gpu:
        import keras; keras.mixed_precision.set_global_policy("mixed_float16")
    log.info(f"DEEPENTXAI-04 | GPU={gpu}")

    X, y, split = load_matrices(cfg)
    dims = {m: X[m].shape[1] for m in INPUT_ORDER}
    tr, te, fold = split["train_idx"], split["test_idx"], split["fold_of_train"]
    log.info(f"dims={dims}  train={len(tr)}  test={len(te)}")

    # --- 1. Optuna hyper-parameter optimisation ------------------------------
    log.info(f"Optuna: {cfg['optuna']['n_trials']} trials ...")
    study = OptunaTuner(cfg, X, y, dims, split, log).run()
    best = {**cfg["model"], **study.best_params}
    joblib.dump(study, os.path.join(cfg.dir_optuna, "study.pkl"))
    save_json({"best_params": study.best_params, "best_val_roc_auc": study.best_value},
              os.path.join(cfg.dir_models, "best_hparams.json"))

    # --- 2. 5-fold cross-validation with the best configuration --------------
    n_folds = int(cfg["split"]["n_folds"])
    cv_rows = []
    for f in range(n_folds):
        va_idx = tr[fold == f]; tr_idx = tr[fold != f]
        model, _ = train_model(X, y, dims, best, cfg, tr_idx, va_idx,
                               epochs=cfg["train"]["epochs"], patience=8, verbose=0)
        p = model.predict(slice_inputs(X, va_idx), verbose=0).ravel()
        m = compute_metrics(y[va_idx], p)
        cv_rows.append(m)
        import keras; keras.backend.clear_session()
        log.info(f"  fold {f}: ROC-AUC {m['roc_auc']:.4f}  MCC {m['mcc']:.4f}")
    cv_mean = {k: float(np.mean([r[k] for r in cv_rows])) for k in cv_rows[0]}
    cv_std = {k: float(np.std([r[k] for r in cv_rows])) for k in cv_rows[0]}
    save_json({"per_fold": cv_rows, "mean": cv_mean, "std": cv_std},
              os.path.join(cfg.dir_metrics, "cv_metrics.json"))
    log.info(f"CV ROC-AUC {cv_mean['roc_auc']:.4f} +/- {cv_std['roc_auc']:.4f}")

    # --- 3. Final model on full train pool -> independent hold-out test -------
    inner_va = tr[fold == 0]; inner_tr = tr[fold != 0]           # val for early stopping
    ckpt = os.path.join(cfg.dir_models, "deepentxai_best.keras")
    model, hist = train_model(X, y, dims, best, cfg, inner_tr, inner_va,
                              epochs=cfg["train"]["epochs"],
                              patience=int(cfg["train"]["early_stopping_patience"]),
                              ckpt_path=ckpt, verbose=0)
    model.save(ckpt)
    p_test = model.predict(slice_inputs(X, te), verbose=0).ravel()
    test_metrics = compute_metrics(y[te], p_test)
    save_json({"test": test_metrics, "cv_mean": cv_mean, "cv_std": cv_std,
               "best_params": study.best_params, "n_test": int(len(te))},
              os.path.join(cfg.dir_metrics, "test_metrics.json"))
    plot_curves(y[te], p_test, cfg.dir_figures, history=hist)
    # cache test predictions for scoring/XAI
    np.savez(os.path.join(cfg.dir_predictions, "test_predictions.npz"),
             y_true=y[te], y_prob=p_test, idx=te)

    print("\n================ DEEPENTXAI-04 complete ================")
    print(f"  best val ROC-AUC (Optuna): {study.best_value:.4f}")
    print(f"  5-fold CV ROC-AUC       : {cv_mean['roc_auc']:.4f} +/- {cv_std['roc_auc']:.4f}")
    print("  --- independent hold-out test ---")
    for k in ("roc_auc", "pr_auc", "accuracy", "balanced_accuracy", "f1", "mcc",
              "recall_sensitivity", "specificity", "cohen_kappa", "brier_score"):
        print(f"    {k:20s}: {test_metrics[k]:.4f}")
    print(f"  model  : {ckpt}")
    print(f"  figures: {cfg.dir_figures}")


if __name__ == "__main__":
    main()


## `04_ensemble_calibrate_threshold.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 08 | Ensemble + Calibration + Threshold Optimisation
=================================================================================
Three LEGITIMATE, leakage-free levers to raise the *accuracy* of the final model
without touching the hold-out test during any tuning step:

  1. 5-model ENSEMBLE      - one model per CV fold; test prob = mean of the five.
                             Averaging cuts variance -> small ROC-AUC gain.
  2. Probability CALIBRATION - isotonic regression fit on OUT-OF-FOLD train
                             predictions (never on test).
  3. THRESHOLD optimisation - the operating point that maximises accuracy on the
                             out-of-fold train predictions, then applied to test.
                             (A 0.5 cut is wrong here: probabilities peak near 0.39.)

Everything tuned on out-of-fold TRAIN predictions; the hold-out test is scored
exactly once, at the end, with the frozen calibrator + threshold.

Outputs : 03_Results/02_Metrics/ensemble_test_metrics.json
          03_Results/03_Model/calibrator.joblib + operating_point.json
          03_Results/05_Predictions/ensemble_test_predictions.npz
Run     : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 08_ensemble_calibrate_threshold.py
=================================================================================
"""
from __future__ import annotations

import json
import os
import sys

import joblib
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.model import INPUT_ORDER
from deepentxai.train import load_matrices, slice_inputs, train_model
from deepentxai.evaluate import compute_metrics


def best_threshold(y, p, objective="accuracy"):
    """Threshold on [0.05,0.95] maximising accuracy or balanced accuracy."""
    from sklearn.metrics import accuracy_score, balanced_accuracy_score
    score = accuracy_score if objective == "accuracy" else balanced_accuracy_score
    grid = np.linspace(0.05, 0.95, 181)
    vals = [score(y, (p >= t).astype(int)) for t in grid]
    j = int(np.argmax(vals))
    return float(grid[j]), float(vals[j])


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB8", cfg.dir_logs)
    import keras

    X, y, split = load_matrices(cfg)
    dims = {m: X[m].shape[1] for m in INPUT_ORDER}
    tr, te, fold = split["train_idx"], split["test_idx"], split["fold_of_train"]
    best = {**cfg["model"], **json.load(open(
        os.path.join(cfg.dir_models, "best_hparams.json")))["best_params"]}
    n_folds = int(cfg["split"]["n_folds"])
    log.info(f"ensemble over {n_folds} folds  dims={dims}  test={len(te)}")

    # --- train one model per fold: OOF train preds + per-fold test preds ------
    oof_prob = np.zeros(len(tr), dtype="float64")
    test_prob_folds = []
    for f in range(n_folds):
        set_seed(cfg.seed + f)                         # decorrelate the ensemble
        va_idx = tr[fold == f]; tr_idx = tr[fold != f]
        model, _ = train_model(X, y, dims, best, cfg, tr_idx, va_idx,
                               epochs=cfg["train"]["epochs"], patience=8, verbose=0)
        oof_prob[fold == f] = model.predict(slice_inputs(X, va_idx), verbose=0).ravel()
        test_prob_folds.append(model.predict(slice_inputs(X, te), verbose=0).ravel())
        model.save(os.path.join(cfg.dir_models, f"ensemble_fold{f}.keras"))   # persist the 5 headline weights
        keras.backend.clear_session()
        log.info(f"  fold {f} trained ({len(tr_idx)} train / {len(va_idx)} val)")

    y_oof = y[tr]
    test_prob = np.mean(test_prob_folds, axis=0)       # ENSEMBLE (mean of 5)
    yte = y[te]

    # --- calibrate on OOF, then choose threshold on calibrated OOF -----------
    from sklearn.isotonic import IsotonicRegression
    iso = IsotonicRegression(out_of_bounds="clip").fit(oof_prob, y_oof)
    joblib.dump(iso, os.path.join(cfg.dir_models, "calibrator.joblib"))
    oof_cal = iso.predict(oof_prob)
    test_cal = iso.predict(test_prob)

    t_acc, oof_acc = best_threshold(y_oof, oof_cal, "accuracy")
    t_bal, oof_bal = best_threshold(y_oof, oof_cal, "balanced")
    save_json({"threshold_accuracy": t_acc, "threshold_balanced": t_bal,
               "oof_accuracy_at_t": oof_acc, "oof_balanced_at_t": oof_bal},
              os.path.join(cfg.dir_models, "operating_point.json"))
    log.info(f"OOF-tuned threshold (max acc) = {t_acc:.3f} -> OOF acc {oof_acc:.4f}")

    # --- score the hold-out test exactly once, four ways ---------------------
    from sklearn.metrics import roc_auc_score
    def acc_at(p, t):
        from sklearn.metrics import accuracy_score
        return accuracy_score(yte, (p >= t).astype(int))

    single = np.load(os.path.join(cfg.dir_predictions, "test_predictions.npz"))["y_prob"]
    variants = {
        "single_model_@0.5":        (single,   0.5),
        "ensemble_@0.5":            (test_prob, 0.5),
        "ensemble_cal_@0.5":        (test_cal,  0.5),
        "ensemble_cal_@tuned":      (test_cal,  t_acc),
    }
    results = {}
    for name, (p, t) in variants.items():
        m = compute_metrics(yte, p, threshold=t)
        results[name] = m

    final = compute_metrics(yte, test_cal, threshold=t_acc)
    final["roc_auc"] = float(roc_auc_score(yte, test_cal))   # calibration is monotone; AUC ~ ensemble
    save_json({"final": final, "variants": results,
               "threshold": t_acc, "n_models": n_folds, "n_test": int(len(te))},
              os.path.join(cfg.dir_metrics, "ensemble_test_metrics.json"))
    np.savez(os.path.join(cfg.dir_predictions, "ensemble_test_predictions.npz"),
             y_true=yte, y_prob=test_prob, y_prob_cal=test_cal, threshold=t_acc,
             oof_true=y_oof, oof_prob=oof_prob, oof_prob_cal=oof_cal)

    print("\n================ DEEPENTXAI-08 complete ================")
    print(f"  tuned threshold (OOF, max-accuracy): {t_acc:.3f}")
    print(f"  {'variant':24s} {'ACC':>7} {'BAL-ACC':>8} {'F1':>7} {'MCC':>7} {'ROC-AUC':>8}")
    for name, m in results.items():
        print(f"  {name:24s} {m['accuracy']:7.4f} {m['balanced_accuracy']:8.4f} "
              f"{m['f1']:7.4f} {m['mcc']:7.4f} {m['roc_auc']:8.4f}")
    print(f"\n  ACCURACY: {results['single_model_@0.5']['accuracy']:.4f} (baseline) "
          f"-> {results['ensemble_cal_@tuned']['accuracy']:.4f} (ensemble+calibrated+tuned)")


if __name__ == "__main__":
    main()


## `05_conformal_selective.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 13 | Conformal Selective Prediction (honest high-confidence)
=================================================================================
The scaffold-disjoint full-coverage ceiling on this phenotypic antibacterial data
is ~0.90 ROC-AUC / ~0.84 accuracy -- limited by the LABELS themselves (replicate
MIC measurements of the same compound disagree ~12% of the time). 98-99% at full
coverage is therefore not physically attainable without leakage.

The legitimate route to a high-accuracy statement is SELECTIVE PREDICTION: let the
model abstain on its least-confident compounds and report accuracy on the ones it
does call, together with the coverage. Everything here is calibrated on OUT-OF-FOLD
train predictions only; the hold-out test is scored once. Two rules:

  (A) confidence cutoff : pick the cutoff tau on OOF so OOF coverage = target c,
      apply tau to test, report the realised test coverage + accuracy.
  (B) Mondrian conformal: class-conditional nonconformity quantile on OOF at level
      alpha; a test point is 'accepted' iff its score <= q_class. Distribution-free
      finite-sample validity, and class-conditional so it does not abstain on one
      class only.

HONESTY RULE baked into the output: the full-coverage row (coverage = 1.0) is
always reported next to every selective row, and every row states its coverage.

Consumes : 03_Results/05_Predictions/ensemble_test_predictions.npz
           (needs oof_true, oof_prob, y_true, y_prob -- produced by script 08)
Outputs  : 03_Results/02_Metrics/conformal_selective.json
           03_Results/01_Figures/risk_coverage.{png,pdf}
Run      : DEEPENT_VARIANT=gapfull DEEPENT_RAW_DIR=... DEEPENT_CONFIG=... \
           ~/miniconda3/envs/ENT/bin/python 13_conformal_selective.py
=================================================================================
"""
from __future__ import annotations

import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import get_logger, save_json


# --- SECTION 1 : selective helpers ------------------------------------------
def decision(p, t=0.5):
    return (p >= t).astype(int)


def confidence(p):
    """Distance from the 0.5 boundary, mapped to [0.5, 1]. Higher = more confident."""
    return np.maximum(p, 1.0 - p)


def acc(y, yhat):
    return float(np.mean(y == yhat)) if len(y) else float("nan")


def selective_by_confidence(oof_y, oof_p, te_y, te_p, coverages, t=0.5):
    """Rule A: cutoff chosen on OOF to hit target coverage, applied to test."""
    oof_conf = confidence(oof_p)
    te_conf = confidence(te_p)
    te_hat = decision(te_p, t)
    rows = []
    for c in coverages:
        # tau = the confidence below which we abstain, set so OOF keeps fraction c
        tau = float(np.quantile(oof_conf, 1.0 - c)) if c < 1.0 else float(oof_conf.min())
        keep = te_conf >= tau
        rows.append({
            "target_coverage": round(float(c), 3),
            "tau": round(tau, 4),
            "test_coverage": round(float(keep.mean()), 4),
            "n_kept": int(keep.sum()),
            "accuracy": round(acc(te_y[keep], te_hat[keep]), 4),
        })
    return rows


def mondrian_conformal(oof_y, oof_p, te_y, te_p, alphas, t=0.5):
    """Rule B: class-conditional (Mondrian) conformal. score = 1 - p(pred class)."""
    oof_hat = decision(oof_p, t)
    oof_pred_prob = np.where(oof_hat == 1, oof_p, 1.0 - oof_p)
    oof_score = 1.0 - oof_pred_prob                      # nonconformity

    te_hat = decision(te_p, t)
    te_pred_prob = np.where(te_hat == 1, te_p, 1.0 - te_p)
    te_score = 1.0 - te_pred_prob

    rows = []
    for a in alphas:
        accept = np.zeros(len(te_y), dtype=bool)
        for k in (0, 1):
            s_k = oof_score[oof_hat == k]
            if len(s_k) == 0:
                continue
            n = len(s_k)
            q = float(np.quantile(s_k, min(1.0, np.ceil((n + 1) * (1 - a)) / n),
                                  method="higher"))
            accept |= (te_hat == k) & (te_score <= q)
        rows.append({
            "alpha": round(float(a), 3),
            "test_coverage": round(float(accept.mean()), 4),
            "n_accepted": int(accept.sum()),
            "accuracy": round(acc(te_y[accept], te_hat[accept]), 4),
        })
    return rows


# --- SECTION 2 : main -------------------------------------------------------
def main() -> None:
    cfg = Config.load()
    log = get_logger("NB13", cfg.dir_logs)

    npz = os.path.join(cfg.dir_predictions, "ensemble_test_predictions.npz")
    d = np.load(npz)
    if "oof_prob" not in d:
        log.error(f"{npz} lacks OOF arrays -- rerun script 08 (patched) first.")
        sys.exit(1)
    oof_y, oof_p = d["oof_true"].astype(int), d["oof_prob"].astype(float)
    te_y, te_p = d["y_true"].astype(int), d["y_prob"].astype(float)
    log.info(f"loaded OOF n={len(oof_y)}  test n={len(te_y)}")

    full = acc(te_y, decision(te_p))
    from sklearn.metrics import roc_auc_score
    auc = float(roc_auc_score(te_y, te_p))

    coverages = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
    ruleA = selective_by_confidence(oof_y, oof_p, te_y, te_p, coverages)
    ruleB = mondrian_conformal(oof_y, oof_p, te_y, te_p,
                               alphas=[0.30, 0.25, 0.20, 0.15, 0.10, 0.05])

    out = {
        "note": ("Selective prediction. Full-coverage row is the honest headline; "
                 "selective rows trade coverage for accuracy. All cutoffs calibrated "
                 "on OOF train predictions only."),
        "full_coverage": {"coverage": 1.0, "accuracy": round(full, 4),
                          "roc_auc": round(auc, 4), "n_test": int(len(te_y))},
        "rule_A_confidence_cutoff": ruleA,
        "rule_B_mondrian_conformal": ruleB,
    }
    save_json(out, os.path.join(cfg.dir_metrics, "conformal_selective.json"))

    # --- risk-coverage figure (titleless) -----------------------------------
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cov = [r["test_coverage"] for r in ruleA]
    ac = [r["accuracy"] for r in ruleA]
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    ax.plot(cov, ac, "-o", color="#4c72b0", lw=1.6, ms=5)
    ax.axhline(full, ls="--", color="grey", lw=1)
    ax.text(0.98, full + 0.004, f"full-coverage acc = {full:.3f}",
            ha="right", va="bottom", fontsize=9, color="grey")
    for r in ruleA:
        if r["target_coverage"] in (0.5, 0.2):
            ax.annotate(f"{r['accuracy']:.3f}\n@cov {r['test_coverage']:.2f}",
                        (r["test_coverage"], r["accuracy"]),
                        textcoords="offset points", xytext=(6, -18), fontsize=8)
    ax.set_xlabel("coverage (fraction of test compounds called)")
    ax.set_ylabel("accuracy on called compounds")
    ax.set_xlim(0, 1.02); ax.invert_xaxis()
    fig.tight_layout()
    figdir = cfg.dir_figures
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(figdir, f"F07_risk_coverage.{ext}"), dpi=300)
    plt.close(fig)

    # --- console summary ----------------------------------------------------
    print("\n================ DEEPENTXAI-13 (conformal selective) ================")
    print(f"  full-coverage:  acc {full:.4f}   ROC-AUC {auc:.4f}   n={len(te_y)}")
    print(f"  {'target-cov':>10} {'test-cov':>9} {'accuracy':>9}   (rule A: OOF confidence cutoff)")
    for r in ruleA:
        print(f"  {r['target_coverage']:10.2f} {r['test_coverage']:9.3f} {r['accuracy']:9.4f}")
    print(f"  {'alpha':>10} {'test-cov':>9} {'accuracy':>9}   (rule B: Mondrian conformal)")
    for r in ruleB:
        print(f"  {r['alpha']:10.2f} {r['test_coverage']:9.3f} {r['accuracy']:9.4f}")
    print("  HONEST HEADLINE: report the selective number ALWAYS beside full-coverage.")


if __name__ == "__main__":
    main()


## `06_explainability.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 06 | Explainable AI
=================================================================================
Explain the trained DEEPENTXAI model with FOUR complementary methods. Compound
scoring & ranking is a separate step (stage 07).

  * Permutation importance  (modality-level + RDKit-descriptor-level)
  * Integrated Gradients    (RDKit branch)
  * SHAP                    (GradientExplainer, RDKit branch; guarded)
  * LIME                    (instance-level; guarded)

Inputs  : 03_Results/03_Model/deepentxai_best.keras + 02_Data features/splits
Outputs : 03_Results/04_Explainability/*  (importances, plots, SHAP/LIME)
Run     : ~/miniconda3/envs/ENT/bin/python 06_explainability.py
=================================================================================
"""
from __future__ import annotations

import json
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.train import load_matrices, slice_inputs
from deepentxai import explain as X_


def _barh(df, value_col, title, path, top=20):
    d = df.head(top)[::-1]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(d.iloc[:, 0], d[value_col], color="#4c72b0", edgecolor="black")
    ax.set_xlabel(value_col); ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB5", cfg.dir_logs)
    import keras

    X, y, split = load_matrices(cfg)
    te = split["test_idx"]
    Xte = slice_inputs(X, te); yte = y[te]
    smiles = np.load(os.path.join(cfg.dir_features_raw, "features.npz"), allow_pickle=True)["smiles"][te]
    names = json.load(open(os.path.join(cfg.dir_features_selected, "selected_rdkit_features.json")))["selected_rdkit_features"]
    model = keras.models.load_model(os.path.join(cfg.dir_models, "deepentxai_best.keras"))
    outdir = cfg.dir_explain
    log.info(f"explaining {len(yte)} hold-out compounds")

    # 1) permutation importance (modality)
    mod = X_.permutation_modality(model, Xte, yte, n_repeats=5, seed=cfg.seed)
    save_json(mod, os.path.join(outdir, "modality_importance.json"))
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(list(mod.keys()), list(mod.values()), color="#dd8452", edgecolor="black")
    ax.set_ylabel("ROC-AUC drop when permuted"); ax.set_title("Modality importance")
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "modality_importance.png"), dpi=300); plt.close(fig)
    log.info(f"modality importance: {mod}")

    # 2) permutation importance (RDKit descriptors)
    perm = X_.permutation_descriptor(model, Xte, yte, names, n_repeats=3, seed=cfg.seed)
    perm.to_csv(os.path.join(outdir, "permutation_descriptor_importance.csv"), index=False)
    _barh(perm, "importance", "Permutation importance (RDKit)", os.path.join(outdir, "permutation_top20.png"))

    # 3) Integrated Gradients (RDKit)
    ig = X_.integrated_gradients_rdkit(model, Xte, names, steps=32, n_samples=300, seed=cfg.seed)
    ig.to_csv(os.path.join(outdir, "integrated_gradients.csv"), index=False)
    _barh(ig, "attribution", "Integrated Gradients (RDKit)", os.path.join(outdir, "integrated_gradients_top20.png"))

    # 4) SHAP + LIME (guarded)
    shap_ok = X_.shap_rdkit(model, Xte, names, outdir, log=log)
    pos = int(np.where(yte == 1)[0][0]); neg = int(np.where(yte == 0)[0][0])
    lime_ok = X_.lime_instances(model, Xte, names, outdir, instances=[pos, neg], log=log)

    print("\n================ DEEPENTXAI-06 (explainability) complete ================")
    print("  modality importance (AUC drop):", {k: round(v, 4) for k, v in mod.items()})
    print("  top-5 RDKit descriptors (permutation):", perm['descriptor'].head(5).tolist())
    print("  top-5 RDKit descriptors (IntGrad)   :", ig['descriptor'].head(5).tolist())
    print(f"  SHAP: {'saved' if shap_ok else 'skipped'} | LIME: {'saved' if lime_ok else 'skipped'}")
    print("  compound scoring + ranking is now stage 07 (07_compound_scoring_and_ranking.py)")


if __name__ == "__main__":
    main()


## `07_compound_scoring_and_ranking.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 07 | Compound Scoring & Ranking (retrospective virtual screen)
=================================================================================
Turns the calibrated 5-fold ENSEMBLE probabilities into a ranked hit list and,
crucially, VALIDATES that ranking on the scaffold-disjoint hold-out where the true
labels are known. A ranking is only scientifically defensible if the actives
actually concentrate at the top; we prove that with the standard early-recognition
metrics used in virtual screening:

  * DEEPENTXAI Score (0-100)  — P(active) blended with boundary confidence.
  * Enrichment Factor EF@x%   — (hit-rate in the top x%) / (overall active rate).
                                EF = 1 is random; ceiling is 1 / active_rate.
  * hit-rate@x% and precision@K.
  * accumulation curve        — actives recovered vs fraction screened.

Scores use the CALIBRATED ENSEMBLE probabilities (the headline model), so "score 90"
is a meaningful probability, and every validation number is computed on TRUE labels
(hence falsifiable). No leakage: the ranking is on the held-out, scaffold-novel set;
nothing here touches training.

Consumes : 03_Results/05_Predictions/ensemble_test_predictions.npz  (from stage 04)
           02_Data/03_Features_Raw/features.npz + 02_Data/05_Splits/split.npz
Outputs  : 03_Results/08_Rankings/compound_ranking.csv   (all hold-out compounds)
           03_Results/08_Rankings/top_hits.csv           (top 100)
           03_Results/02_Metrics/screening_metrics.json  (EF / hit-rate / precision@K)
           03_Results/01_Figures/enrichment_curve.{png,pdf}
Run      : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 07_compound_scoring_and_ranking.py
=================================================================================
"""
from __future__ import annotations

import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import get_logger, save_json
from deepentxai.scoring import (rank_compounds, enrichment, precision_at_k,
                                accumulation_curve, deepentxai_score)


def main() -> None:
    cfg = Config.load()
    log = get_logger("NB7scr", cfg.dir_logs)

    # --- load the calibrated ENSEMBLE predictions + aligned SMILES -----------
    pred = np.load(os.path.join(cfg.dir_predictions, "ensemble_test_predictions.npz"))
    y_true = pred["y_true"].astype(int)
    prob = (pred["y_prob_cal"] if "y_prob_cal" in pred else pred["y_prob"]).astype(float)

    split = np.load(os.path.join(cfg.dir_splits, "split.npz"))
    te = split["test_idx"]
    smiles = np.load(os.path.join(cfg.dir_features_raw, "features.npz"),
                     allow_pickle=True)["smiles"][te]
    assert len(smiles) == len(y_true) == len(prob), "test alignment mismatch"
    log.info(f"scoring {len(y_true)} scaffold-disjoint hold-out compounds "
             f"({int(y_true.sum())} active)")

    # --- ranking -------------------------------------------------------------
    ranking = rank_compounds(np.asarray(smiles), prob, y_true)
    ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
    ranking.to_csv(os.path.join(cfg.dir_rankings, "compound_ranking.csv"), index=False)
    ranking.head(100).to_csv(os.path.join(cfg.dir_rankings, "top_hits.csv"), index=False)

    # --- retrospective virtual-screening validation --------------------------
    ef = enrichment(y_true, prob, fractions=(0.01, 0.05, 0.10, 0.20))
    pk = precision_at_k(y_true, prob, ks=(10, 25, 50, 100, 250, 500))
    from sklearn.metrics import roc_auc_score, average_precision_score
    metrics = {
        "driver": "calibrated_ensemble_probability",
        "ranking_roc_auc": round(float(roc_auc_score(y_true, prob)), 4),
        "ranking_pr_auc": round(float(average_precision_score(y_true, prob)), 4),
        "enrichment": ef,
        "precision_at_k": pk,
        "note": ("Retrospective virtual screen on the scaffold-disjoint hold-out. "
                 "EF=1 is random; ceiling is 1/active_rate = "
                 f"{ef['max_EF']}. All figures computed on true labels."),
    }
    save_json(metrics, os.path.join(cfg.dir_metrics, "screening_metrics.json"))

    # --- accumulation (enrichment) curve, titleless --------------------------
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fx, fy = accumulation_curve(y_true, prob)
    base = ef["active_rate"]
    fig, ax = plt.subplots(figsize=(5.4, 4.8))
    ax.plot(fx, fy, "-", color="#4c72b0", lw=1.8, label="DEEPENTXAI ranking")
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="random")
    # ideal early-recognition curve (all actives first)
    ideal_x = [0, base, 1]; ideal_y = [0, 1, 1]
    ax.plot(ideal_x, ideal_y, ":", color="#55a868", lw=1.2, label="ideal")
    ax.set_xlabel("fraction of compounds screened (ranked by score)")
    ax.set_ylabel("fraction of true actives recovered")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend(loc="lower right", fontsize=9)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(cfg.dir_figures, f"F08_enrichment_curve.{ext}"), dpi=300)
    plt.close(fig)

    # --- console summary -----------------------------------------------------
    print("\n================ DEEPENTXAI-07 (scoring & ranking) ================")
    print(f"  compounds ranked: {ef['n_total']}  |  actives: {ef['n_active']} "
          f"({ef['active_rate']*100:.1f}%)  |  max possible EF: {ef['max_EF']}")
    print(f"  ranking ROC-AUC {metrics['ranking_roc_auc']}  PR-AUC {metrics['ranking_pr_auc']}")
    print(f"  {'top-x%':>8} {'k':>6} {'hits':>6} {'hit-rate':>9} {'EF':>7}")
    for f, r in ef["by_fraction"].items():
        print(f"  {f:>8} {r['k']:>6} {r['hits']:>6} {r['hit_rate']:>9.3f} {r['EF']:>7.2f}")
    print("  precision@K:", {k: v for k, v in pk.items()})
    print(f"  top hit score: {ranking['deepentxai_score'].iloc[0]:.1f} "
          f"(rank 1, observed {ranking['observed'].iloc[0]})")


if __name__ == "__main__":
    main()


## `08_reports_and_figures.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 07 | Publication-Quality Figures & Final Reports
=================================================================================
Consolidate every artifact from Scripts 01-05 into manuscript-ready deliverables:
a metrics table, a summary figure, and a full markdown report describing the
dataset, features, architecture, validation, explainability and top compounds.

Outputs : 03_Results/02_Metrics/publication_metrics_table.csv
          03_Results/01_Figures/07_summary.png
          03_Results/DEEPENTXAI_Report.md
Run     : ~/miniconda3/envs/ENT/bin/python 07_reports_and_figures.py
=================================================================================
"""
from __future__ import annotations

import json
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import get_logger


def _load(path):
    return json.load(open(path)) if os.path.exists(path) else {}


def main() -> None:
    cfg = Config.load()
    log = get_logger("NB7", cfg.dir_logs)
    M = cfg.dir_metrics
    nb1 = _load(os.path.join(M, "nb1_preprocessing_report.json"))
    nb2 = _load(os.path.join(M, "nb2_feature_report.json"))
    test = _load(os.path.join(M, "test_metrics.json"))
    cv = _load(os.path.join(M, "cv_metrics.json"))
    mod = _load(os.path.join(cfg.dir_explain, "modality_importance.json"))

    # Headline = the 5-fold CNN-LSTM ENSEMBLE (stage 04) if available, else the
    # single tuned hold-out model (stage 03). Keeps the report consistent with the
    # ensemble numbers quoted everywhere else.
    ens = _load(os.path.join(M, "ensemble_test_metrics.json"))
    tm = ens.get("variants", {}).get("ensemble_@0.5") or test.get("test", {})
    headline = "5-fold CNN-LSTM ensemble" if ens.get("variants", {}).get("ensemble_@0.5") else "single hold-out model"
    cvm, cvs = cv.get("mean", {}), cv.get("std", {})

    # --- metrics table -------------------------------------------------------
    order = ["accuracy", "balanced_accuracy", "precision", "recall_sensitivity",
             "specificity", "f1", "roc_auc", "pr_auc", "mcc", "cohen_kappa",
             "log_loss", "brier_score"]
    rows = [{"metric": k,
             "hold_out_test": round(tm.get(k, float('nan')), 4),
             "cv_mean": round(cvm.get(k, float('nan')), 4),
             "cv_std": round(cvs.get(k, float('nan')), 4)} for k in order if k in tm]
    tbl = pd.DataFrame(rows)
    tbl.to_csv(os.path.join(M, "publication_metrics_table.csv"), index=False)

    # --- summary figure ------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 5))
    keys = ["roc_auc", "pr_auc", "accuracy", "balanced_accuracy", "f1", "mcc"]
    vals = [tm.get(k, 0) for k in keys]
    ax.bar(keys, vals, color="#4c72b0", edgecolor="black")
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")
    ax.set_ylim(0, 1); ax.set_ylabel("score")
    ax.set_title("DEEPENTXAI — independent hold-out performance")
    fig.tight_layout(); fig.savefig(os.path.join(cfg.dir_figures, "F12_summary.png"), dpi=300); plt.close(fig)

    # --- markdown report -----------------------------------------------------
    top_desc = []
    p = os.path.join(cfg.dir_explain, "permutation_descriptor_importance.csv")
    if os.path.exists(p):
        top_desc = pd.read_csv(p)["descriptor"].head(10).tolist()
    rk = os.path.join(cfg.dir_rankings, "top_hits.csv")
    n_top = len(pd.read_csv(rk)) if os.path.exists(rk) else 0

    def m(k): return f"{tm.get(k, float('nan')):.3f}"

    rep = f"""# DEEPENTXAI — Final Report

Explainable multimodal fusion **CNN-LSTM** for anti-*Enterobacteriaceae* compound
bioactivity prediction (binary Active/Inactive). Leakage-free, scaffold-disjoint,
reproducible.

## 1. Dataset
- Sources: ChEMBL + PubChem BioAssay (official APIs)
- Unique standardised compounds: **{nb1.get('final', {}).get('unique_compounds', '—')}**
  (active {nb1.get('final', {}).get('active', '—')} / inactive {nb1.get('final', {}).get('inactive', '—')})
- Standardisation: salts stripped, neutralised, canonicalised; conflicts resolved by majority vote
- Activity label: IC50/EC50/Ki/Kd ≤ {cfg['labeling']['affinity_threshold_uM']} µM = Active

## 2. Features
- Morgan ECFP4 (2048) · RDKit descriptors ({nb2.get('selection', {}).get('from_total_rdkit', '—')} → **{nb2.get('selection', {}).get('n_selected_rdkit', '—')}** selected) · MACCS (167) · ChemBERTa (768)
- Selection: variance → correlation → Mutual Information → RFE, **fit on train only**
- Split: **scaffold-disjoint** hold-out ({nb2.get('split', {}).get('n_test', '—')}) + {nb2.get('split', {}).get('n_folds', 5)}-fold CV on {nb2.get('split', {}).get('n_train', '—')} train (disjoint = {nb2.get('split', {}).get('holdout_scaffold_disjoint', '—')})

## 3. Model
- Multimodal fusion CNN-LSTM: per-modality encoders → fusion → BatchNorm → Residual CNN → Channel Attention → BiLSTM → Residual Dense → Sigmoid
- Hyper-parameters selected by **Optuna** ({cfg['optuna']['n_trials']} trials)

## 4. Performance
| | ROC-AUC | PR-AUC | Accuracy | Bal-Acc | F1 | MCC |
|---|---|---|---|---|---|---|
| **Hold-out test** | {m('roc_auc')} | {m('pr_auc')} | {m('accuracy')} | {m('balanced_accuracy')} | {m('f1')} | {m('mcc')} |
| **5-fold CV** | {cvm.get('roc_auc', float('nan')):.3f} ± {cvs.get('roc_auc', 0):.3f} | {cvm.get('pr_auc', float('nan')):.3f} | {cvm.get('accuracy', float('nan')):.3f} | {cvm.get('balanced_accuracy', float('nan')):.3f} | {cvm.get('f1', float('nan')):.3f} | {cvm.get('mcc', float('nan')):.3f} |

Also reported: sensitivity {m('recall_sensitivity')}, specificity {m('specificity')},
Cohen's κ {m('cohen_kappa')}, log-loss {m('log_loss')}, Brier {m('brier_score')}.
Curves: `03_Results/01_Figures/{{roc,pr,confusion,calibration,learning}}_curve.png`.

## 5. Explainability
- Modality importance (permutation AUC-drop): { {k: round(v,4) for k,v in mod.items()} }
- Top RDKit descriptors: {', '.join(top_desc) if top_desc else '—'}
- SHAP summary, Integrated-Gradients and LIME plots in `03_Results/04_Explainability/`

## 6. Compound scoring
- **DEEPENTXAI Score** (0–100) ranks compounds by confidence-weighted P(active);
  top {n_top} hits exported to `03_Results/08_Rankings/top_hits.csv`.

*Generated by DEEPENTXAI Script 07.*
"""
    rep_path = os.path.join(cfg.root, "03_Results", "DEEPENTXAI_Report.md")
    with open(rep_path, "w") as fh:
        fh.write(rep)

    print("\n================ DEEPENTXAI-07 complete ================")
    print(f"  metrics table : {os.path.join(M, 'publication_metrics_table.csv')}")
    print(f"  summary figure: {os.path.join(cfg.dir_figures, 'F12_summary.png')}")
    print(f"  final report  : {rep_path}")
    print(tbl.to_string(index=False))


if __name__ == "__main__":
    main()


## `09_baselines_and_stacking.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 09 | Stacked Ensemble (CNN-LSTM + XGBoost + RandomForest)
=================================================================================
Combine three complementary base learners with a logistic meta-learner, trained
ONLY on out-of-fold (OOF) train predictions -> leakage-free. The hold-out test is
scored once at the end. Base learners disagree in different regions, so stacking
typically adds ~0.01-0.03 ROC-AUC over any single model.

  base 1 : DEEPENTXAI CNN-LSTM  (multimodal, per-fold, OOF + test)
  base 2 : XGBoost              (flat 3083-dim feature vector)
  base 3 : RandomForest         (flat 3083-dim feature vector)
  meta   : LogisticRegression on the three OOF probability columns

Outputs : 03_Results/02_Metrics/stacked_test_metrics.json
          03_Results/03_Model/stack_meta.joblib
Run     : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 09_stacked_ensemble.py
=================================================================================
"""
from __future__ import annotations

import json
import os
import sys

import joblib
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.model import INPUT_ORDER
from deepentxai.train import load_matrices, slice_inputs, train_model
from deepentxai.evaluate import compute_metrics


def best_threshold(y, p):
    from sklearn.metrics import accuracy_score
    grid = np.linspace(0.05, 0.95, 181)
    vals = [accuracy_score(y, (p >= t).astype(int)) for t in grid]
    j = int(np.argmax(vals))
    return float(grid[j]), float(vals[j])


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB9", cfg.dir_logs)
    import keras
    from xgboost import XGBClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score

    X, y, split = load_matrices(cfg)
    dims = {m: X[m].shape[1] for m in INPUT_ORDER}
    tr, te, fold = split["train_idx"], split["test_idx"], split["fold_of_train"]
    best = {**cfg["model"], **json.load(open(
        os.path.join(cfg.dir_models, "best_hparams.json")))["best_params"]}
    n_folds = int(cfg["split"]["n_folds"])

    # flat feature matrix for the tree learners (all modalities concatenated)
    Xflat = np.concatenate([X[m] for m in INPUT_ORDER], axis=1)
    log.info(f"stacking: flat dim={Xflat.shape[1]}  train={len(tr)}  test={len(te)}")

    y_oof = y[tr]
    oof = {k: np.zeros(len(tr)) for k in ("cnn", "xgb", "rf")}
    tst = {k: [] for k in ("cnn", "xgb", "rf")}

    for f in range(n_folds):
        tr_idx = tr[fold != f]; va_idx = tr[fold == f]     # GLOBAL indices into X / Xflat
        pos_va = np.where(fold == f)[0]                     # positions into the train-pool arrays (oof)

        # base 1: CNN-LSTM
        set_seed(cfg.seed + f)
        model, _ = train_model(X, y, dims, best, cfg, tr_idx, va_idx,
                               epochs=cfg["train"]["epochs"], patience=8, verbose=0)
        oof["cnn"][pos_va] = model.predict(slice_inputs(X, va_idx), verbose=0).ravel()
        tst["cnn"].append(model.predict(slice_inputs(X, te), verbose=0).ravel())
        keras.backend.clear_session()

        # base 2 + 3: trees on the flat vector (index Xflat with GLOBAL indices)
        xgb = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.6, eval_metric="logloss",
                            n_jobs=8, random_state=cfg.seed + f, tree_method="hist")
        xgb.fit(Xflat[tr_idx], y[tr_idx])
        oof["xgb"][pos_va] = xgb.predict_proba(Xflat[va_idx])[:, 1]
        tst["xgb"].append(xgb.predict_proba(Xflat[te])[:, 1])

        rf = RandomForestClassifier(n_estimators=500, max_depth=None, n_jobs=8,
                                    max_features="sqrt", random_state=cfg.seed + f)
        rf.fit(Xflat[tr_idx], y[tr_idx])
        oof["rf"][pos_va] = rf.predict_proba(Xflat[va_idx])[:, 1]
        tst["rf"].append(rf.predict_proba(Xflat[te])[:, 1])
        log.info(f"  fold {f}: cnn/xgb/rf OOF done")

    yte = y[te]
    test_base = {k: np.mean(v, axis=0) for k, v in tst.items()}     # mean over folds
    Z_oof = np.column_stack([oof["cnn"], oof["xgb"], oof["rf"]])
    Z_te = np.column_stack([test_base["cnn"], test_base["xgb"], test_base["rf"]])

    # meta-learner on OOF only
    meta = LogisticRegression(max_iter=1000).fit(Z_oof, y_oof)
    joblib.dump(meta, os.path.join(cfg.dir_models, "stack_meta.joblib"))
    p_oof = meta.predict_proba(Z_oof)[:, 1]
    p_te = meta.predict_proba(Z_te)[:, 1]
    t_acc, oof_acc = best_threshold(y_oof, p_oof)
    log.info(f"meta coefs (cnn,xgb,rf)={meta.coef_.ravel().round(3).tolist()}  "
             f"tuned t={t_acc:.3f} OOF acc={oof_acc:.4f}")

    # score hold-out once
    results = {}
    for name, p, t in [
        ("cnn_lstm_@0.5",  test_base["cnn"], 0.5),
        ("xgboost_@0.5",   test_base["xgb"], 0.5),
        ("randomforest_@0.5", test_base["rf"], 0.5),
        ("stack_@0.5",     p_te, 0.5),
        ("stack_@tuned",   p_te, t_acc),
    ]:
        m = compute_metrics(yte, p, threshold=t)
        m["roc_auc"] = float(roc_auc_score(yte, p))
        results[name] = m

    save_json({"variants": results, "meta_coef": meta.coef_.ravel().tolist(),
               "threshold": t_acc, "n_test": int(len(te))},
              os.path.join(cfg.dir_metrics, "stacked_test_metrics.json"))

    print("\n================ DEEPENTXAI-09 complete ================")
    print(f"  {'model':22s} {'ACC':>7} {'BAL-ACC':>8} {'F1':>7} {'MCC':>7} {'ROC-AUC':>8} {'PR-AUC':>7}")
    for name, m in results.items():
        print(f"  {name:22s} {m['accuracy']:7.4f} {m['balanced_accuracy']:8.4f} "
              f"{m['f1']:7.4f} {m['mcc']:7.4f} {m['roc_auc']:8.4f} {m['pr_auc']:7.4f}")


if __name__ == "__main__":
    main()


## `10_regression_ecoli_mic.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 11 | E. coli whole-cell MIC REGRESSION (pMIC)
=================================================================================
Reframe as a REGRESSION QSAR on a single coherent endpoint: predict pMIC of
whole-cell E. coli growth inhibition. Single species, single endpoint (fixes the
affinity/MIC pooling critique), exact '=' measurements only, censored '>' handled
by exclusion, per-structure median (robust to 2-fold assay noise), scaffold-
disjoint split (no leakage).

Multimodal fusion CNN-LSTM with a LINEAR head + MSE. Reports R2/RMSE/MAE/Pearson/
Spearman with bootstrap 95% CIs, RF/XGB regressor baselines, and a 5-model
ensemble. All transforms fit on TRAIN only.

Input   : 02_Data/06_Regression/labelled_reg.csv  (smiles, pmic, n_meas)
Outputs : 03_Results/02_Metrics/ecolimic_regression.json + 03_Results/01_Figures/regression_scatter.*
Run     : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 11_regression_ecoli_mic.py
=================================================================================
"""
from __future__ import annotations
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.features import FeatureGenerator

INPUT_ORDER = ["morgan", "rdkit", "maccs", "chemberta"]


# ---------------- scaffold-disjoint split (whole Murcko groups) --------------
def scaffold_split(scaffolds, test_frac=0.15, n_folds=5, seed=42):
    rng = np.random.RandomState(seed)
    groups = {}
    for i, s in enumerate(scaffolds):
        groups.setdefault(s, []).append(i)
    keys = list(groups.keys()); rng.shuffle(keys)
    n = len(scaffolds); n_test = int(test_frac * n)
    test = []
    for k in keys:
        if len(test) >= n_test: break
        test += groups[k]
    test = np.array(sorted(test)); train = np.array(sorted(set(range(n)) - set(test)))
    # scaffold-aware folds on train
    tr_keys = [k for k in keys if all(i in set(train) for i in groups[k])]
    fold_of = np.full(len(train), -1); pos = {g: p for p, g in enumerate(train)}
    tk = [k for k in keys if k in set(scaffolds[train])]
    rng.shuffle(tk)
    for j, k in enumerate(tk):
        for i in groups[k]:
            if i in pos: fold_of[pos[i]] = j % n_folds
    fold_of[fold_of == -1] = 0
    return train, test, fold_of


def metrics(y, p):
    from scipy.stats import pearsonr, spearmanr
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    return {"r2": float(r2_score(y, p)), "rmse": float(np.sqrt(mean_squared_error(y, p))),
            "mae": float(mean_absolute_error(y, p)),
            "pearson": float(pearsonr(y, p)[0]), "spearman": float(spearmanr(y, p)[0])}


def boot_ci(y, p, fn, n=2000, seed=42):
    rng = np.random.RandomState(seed); vals = []
    for _ in range(n):
        idx = rng.randint(0, len(y), len(y)); vals.append(fn(y[idx], p[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def build_reg_model(dims, hp):
    import keras
    from keras import layers as L
    ins, encs = [], []
    for m in INPUT_ORDER:
        x = keras.Input(shape=(dims[m],), name=m); ins.append(x)
        encs.append(L.Dense(hp["embed_dim"], activation=hp["activation"])(x))
    h = L.Concatenate()(encs); h = L.BatchNormalization()(h)
    h = L.Dense(64 * 8, activation=hp["activation"])(h); h = L.Reshape((64, 8))(h)
    for _ in range(hp["cnn_blocks"]):
        r = h
        h = L.Conv1D(hp["cnn_filters"], hp["cnn_kernel"], padding="same", activation=hp["activation"])(h)
        h = L.Conv1D(8, hp["cnn_kernel"], padding="same")(h)
        h = L.Add()([r, h]); h = L.Activation(hp["activation"])(h)
    # squeeze-excite channel attention
    s = L.GlobalAveragePooling1D()(h); s = L.Dense(8 // 2 or 1, activation="relu")(s)
    s = L.Dense(8, activation="sigmoid")(s); h = L.Multiply()([h, L.Reshape((1, 8))(s)])
    h = L.Bidirectional(L.LSTM(hp["lstm_units"]))(h)
    d = L.Dense(hp["dense_units"], activation=hp["activation"])(h)
    d = L.Dense(hp["dense_units"] // 2, activation=hp["activation"])(d); d = L.Add()([d, d])
    d = L.Dropout(hp["dropout"])(d)
    out = L.Dense(1, dtype="float32")(d)                       # LINEAR head
    mdl = keras.Model(ins, out)
    mdl.compile(optimizer=keras.optimizers.Nadam(hp["learning_rate"], clipnorm=1.0), loss="mse")
    return mdl


def main():
    cfg = Config.load(); set_seed(cfg.seed)
    log = get_logger("NB11", cfg.dir_logs)
    import keras
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_selection import mutual_info_regression
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestRegressor
    from rdkit import Chem

    df = pd.read_csv(os.path.join(cfg.root, "02_Data", "06_Regression", "labelled_reg.csv"))
    y = df["pmic"].values.astype("float32"); smiles = df["smiles"].tolist()
    log.info(f"{len(df)} compounds  pMIC mean={y.mean():.2f} std={y.std():.2f}")

    fg = FeatureGenerator(cfg["features"])
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    ok = [i for i, m in enumerate(mols) if m is not None]
    mols = [mols[i] for i in ok]; smiles = [smiles[i] for i in ok]; y = y[ok]
    morgan = np.stack([fg.morgan(m) for m in mols])
    maccs = np.stack([fg.maccs(m) for m in mols])
    rdk = np.array([fg.rdkit(m) for m in mols], dtype=np.float64)
    rdk = np.where(np.isfinite(rdk), rdk, np.nan).astype("float32")
    scaf = np.array([fg.scaffold(m) or smiles[i] for i, m in enumerate(mols)])  # M2 fix: fallback
    log.info("computing ChemBERTa embeddings ...")
    cberta = fg.chemberta(smiles, log=log)

    tr, te, fold = scaffold_split(scaf, seed=cfg.seed)
    # scaffold-disjoint sanity
    assert len(set(scaf[tr]) & set(scaf[te])) == 0, "scaffold leak!"
    log.info(f"train={len(tr)} test={len(te)} scaffold-disjoint=True")

    # RDKit: impute+select(MI regression, top100)+scale  — TRAIN ONLY
    imp = SimpleImputer(strategy="median").fit(rdk[tr]); rdk_i = imp.transform(rdk)
    mi = mutual_info_regression(rdk_i[tr], y[tr], random_state=cfg.seed)
    keep = np.argsort(mi)[::-1][:100]
    sc_r = StandardScaler().fit(rdk_i[tr][:, keep]); rdk_s = sc_r.transform(rdk_i[:, keep])
    sc_b = StandardScaler().fit(cberta[tr]); cb_s = sc_b.transform(cberta)
    X = {"morgan": morgan.astype("float32"), "rdkit": rdk_s.astype("float32"),
         "maccs": maccs.astype("float32"), "chemberta": cb_s.astype("float32")}
    dims = {m: X[m].shape[1] for m in INPUT_ORDER}

    hp = {"embed_dim": 128, "cnn_blocks": 2, "cnn_filters": 64, "cnn_kernel": 3,
          "lstm_units": 128, "dense_units": 256, "dropout": 0.3, "activation": "gelu",
          "learning_rate": 5e-4}

    def sl(idx): return [X[m][idx] for m in INPUT_ORDER]

    # 5-model scaffold-fold ensemble
    test_preds = []
    for f in range(5):
        set_seed(cfg.seed + f)
        tri = tr[fold != f]; vai = tr[fold == f]
        mdl = build_reg_model(dims, hp)
        cb = [keras.callbacks.EarlyStopping("val_loss", patience=10, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau("val_loss", patience=5, factor=0.5, min_lr=1e-6)]
        mdl.fit(sl(tri), y[tri], validation_data=(sl(vai), y[vai]),
                epochs=120, batch_size=64, callbacks=cb, verbose=0)
        test_preds.append(mdl.predict(sl(te), verbose=0).ravel())
        keras.backend.clear_session(); log.info(f"  fold {f} trained")
    p_ens = np.mean(test_preds, axis=0)

    # baselines on concatenated features
    Xflat = np.concatenate([X[m] for m in INPUT_ORDER], axis=1)
    rf = RandomForestRegressor(n_estimators=400, n_jobs=8, random_state=cfg.seed).fit(Xflat[tr], y[tr])
    p_rf = rf.predict(Xflat[te])
    try:
        from xgboost import XGBRegressor
        xgb = XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                           colsample_bytree=0.6, n_jobs=8, random_state=cfg.seed, tree_method="hist").fit(Xflat[tr], y[tr])
        p_xgb = xgb.predict(Xflat[te])
    except Exception:
        p_xgb = None

    yte = y[te]
    out = {"n_total": int(len(y)), "n_train": int(len(tr)), "n_test": int(len(te)),
           "pmic_std": float(y.std())}
    for name, p in [("cnn_lstm_ensemble", p_ens), ("randomforest", p_rf)] + ([("xgboost", p_xgb)] if p_xgb is not None else []):
        m = metrics(yte, p)
        m["r2_ci"] = boot_ci(yte, p, lambda a, b: __import__("sklearn.metrics", fromlist=["r2_score"]).r2_score(a, b))
        m["pearson_ci"] = boot_ci(yte, p, lambda a, b: __import__("scipy.stats", fromlist=["pearsonr"]).pearsonr(a, b)[0])
        out[name] = m
    save_json(out, os.path.join(cfg.dir_metrics, "ecolimic_regression.json"))

    # titleless scatter: predicted vs measured pMIC (ensemble)
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.scatter(yte, p_ens, s=8, alpha=0.35, color="#4c72b0", edgecolors="none")
    lo, hi = min(yte.min(), p_ens.min()), max(yte.max(), p_ens.max())
    ax.plot([lo, hi], [lo, hi], "--", color="grey", lw=1)
    ax.set_xlabel("measured pMIC"); ax.set_ylabel("predicted pMIC")
    m = out["cnn_lstm_ensemble"]
    ax.text(0.05, 0.92, f"R$^2$={m['r2']:.3f}  r={m['pearson']:.3f}\nRMSE={m['rmse']:.2f}",
            transform=ax.transAxes, fontsize=10, va="top")
    figdir = cfg._mk("03_Results", "01_Figures")
    fig.tight_layout()
    for ext in ("png", "pdf"): fig.savefig(os.path.join(figdir, f"F11_regression_scatter.{ext}"), dpi=300)
    plt.close(fig)

    print("\n================ DEEPENTXAI-11 (E. coli MIC regression) ================")
    print(f"  {len(y)} compounds | train {len(tr)} / test {len(te)} (scaffold-disjoint)")
    print(f"  {'model':20s} {'R2':>7} {'Pearson':>8} {'Spearman':>9} {'RMSE':>7} {'MAE':>7}")
    for name in ("cnn_lstm_ensemble", "randomforest", "xgboost"):
        if name in out:
            m = out[name]
            print(f"  {name:20s} {m['r2']:7.3f} {m['pearson']:8.3f} {m['spearman']:9.3f} {m['rmse']:7.3f} {m['mae']:7.3f}")
    m = out["cnn_lstm_ensemble"]
    print(f"  ensemble R2 95% CI {m['r2_ci']}  Pearson 95% CI {m['pearson_ci']}")


if __name__ == "__main__":
    main()


## `11_leakage_demonstration.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI | Script 10 | LEAKAGE DEMONSTRATION  (random split vs scaffold split)
=================================================================================
!!! DIAGNOSTIC ONLY -- THE RANDOM-SPLIT NUMBER IS NOT A VALID PREDICTIVE RESULT !!!

Same features, same CNN-LSTM, same 5-model ensemble as the honest pipeline. The
ONLY change is the train/test split:

    scaffold-disjoint  -> honest generalisation to NEW chemistry  (~0.79 acc)
    random  (row-wise) -> near-duplicate analogs of the SAME scaffold leak into
                          both train and test, so the model "recognises" test
                          molecules it effectively already saw            (inflated)

The gap between the two is the data leakage that inflated accuracy figures in the
literature are (often unknowingly) reporting. Report the SCAFFOLD number.

Outputs : 03_Results/02_Metrics/leakage_demo.json
Run     : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 10_leakage_demo_random_split.py
=================================================================================
"""
from __future__ import annotations

import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.model import INPUT_ORDER
from deepentxai.train import slice_inputs, train_model
from deepentxai.evaluate import compute_metrics


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB10", cfg.dir_logs)
    import keras
    from sklearn.model_selection import train_test_split, StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score

    feats = np.load(os.path.join(cfg.dir_features_raw, "features.npz"), allow_pickle=True)
    rsel = np.load(os.path.join(cfg.dir_features_selected, "rdkit_selected.npz"))
    y = feats["y"].astype(int)
    n = len(y)
    best = {**cfg["model"], **json.load(open(
        os.path.join(cfg.dir_models, "best_hparams.json")))["best_params"]}
    n_folds = int(cfg["split"]["n_folds"])
    test_size = float(cfg["split"]["test_size"])

    # ---- RANDOM stratified split (row-wise) -- THIS IS THE LEAKY SETUP -------
    all_idx = np.arange(n)
    tr, te = train_test_split(all_idx, test_size=test_size, random_state=cfg.seed,
                              stratify=y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=cfg.seed)
    fold = np.empty(len(tr), dtype=int)
    for fi, (_, va_pos) in enumerate(skf.split(tr, y[tr])):
        fold[va_pos] = fi

    # chemberta scaled on the (random) train pool only
    sb = StandardScaler().fit(feats["X_chemberta"][tr])
    X = {"morgan": feats["X_morgan"].astype("float32"),
         "rdkit": rsel["X_rdkit_selected"].astype("float32"),
         "maccs": feats["X_maccs"].astype("float32"),
         "chemberta": sb.transform(feats["X_chemberta"]).astype("float32")}
    dims = {m: X[m].shape[1] for m in INPUT_ORDER}
    log.info(f"RANDOM split: train={len(tr)} test={len(te)}  (LEAKAGE DEMO)")

    # ---- same 5-model ensemble as the honest run ----------------------------
    test_probs = []
    for f in range(n_folds):
        set_seed(cfg.seed + f)
        tr_idx = tr[fold != f]; va_idx = tr[fold == f]
        model, _ = train_model(X, y, dims, best, cfg, tr_idx, va_idx,
                               epochs=cfg["train"]["epochs"], patience=8, verbose=0)
        test_probs.append(model.predict(slice_inputs(X, te), verbose=0).ravel())
        keras.backend.clear_session()
        log.info(f"  fold {f} trained")
    p_te = np.mean(test_probs, axis=0)
    yte = y[te]
    m = compute_metrics(yte, p_te)
    m["roc_auc"] = float(roc_auc_score(yte, p_te))

    # ---- honest scaffold ensemble for side-by-side --------------------------
    scaf = {}
    ens_path = os.path.join(cfg.dir_metrics, "ensemble_test_metrics.json")
    if os.path.exists(ens_path):
        scaf = json.load(open(ens_path)).get("variants", {}).get("ensemble_@0.5", {})

    save_json({"WARNING": "random-split result is a LEAKAGE ARTIFACT, not a valid predictive estimate",
               "random_split_LEAKY": m, "scaffold_split_HONEST": scaf},
              os.path.join(cfg.dir_metrics, "leakage_demo.json"))

    print("\n" + "=" * 66)
    print("  LEAKAGE DEMONSTRATION  (same model, same data, split changed)")
    print("=" * 66)
    print(f"  {'metric':14s} {'RANDOM (leaky)':>16} {'SCAFFOLD (honest)':>18}")
    for k in ("accuracy", "balanced_accuracy", "f1", "mcc", "roc_auc", "pr_auc"):
        rv = m.get(k, float('nan')); sv = scaf.get(k, float('nan'))
        print(f"  {k:14s} {rv:16.4f} {sv:18.4f}")
    print("=" * 66)
    print("  >>> The RANDOM column is INFLATED BY LEAKAGE. Do NOT report it as a")
    print("  >>> predictive result. The SCAFFOLD column is the real performance.")
    print("=" * 66)


if __name__ == "__main__":
    main()


## `12_assay_source_bias.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 12 | Assay / Source-Bias Analysis   (reviewer R2.6)
=================================================================================
Reviewer concern: if actives and inactives come from different assays/sources, the
model might learn a SOURCE artefact instead of true biological activity.

This stage tests that on the trained headline model WITHOUT retraining anything, so
the reported accuracy is untouched. It answers three questions:

  (a) Are the classes confounded with source?  -> active-rate per source.
  (b) Can the model tell sources apart by chance?  -> class balance per source.
  (c) THE decisive test: does restricting the hold-out test to a SINGLE source
      change the ROC-AUC?  If AUC is stable within each source, the signal is
      biological, not a source artefact.

Consumes : 03_Results/05_Predictions/ensemble_test_predictions.npz (from stage 04)
           02_Data/02_Processed/labelled.csv (the 'sources' column)
           02_Data/05_Splits/split.npz (test_idx, to align sources to predictions)
Outputs  : 03_Results/02_Metrics/assay_source_bias.json
Run      : ~/miniconda3/envs/ENT/bin/python 12_assay_source_bias.py
=================================================================================
"""
from __future__ import annotations
import os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import get_logger, save_json


def main() -> None:
    cfg = Config.load()
    log = get_logger("NB12", cfg.dir_logs)
    from sklearn.metrics import roc_auc_score, average_precision_score

    df = pd.read_csv(os.path.join(cfg.dir_processed, "labelled.csv"))
    split = np.load(os.path.join(cfg.dir_splits, "split.npz"))
    te = split["test_idx"]
    pred = np.load(os.path.join(cfg.dir_predictions, "ensemble_test_predictions.npz"))
    y = pred["y_true"].astype(int)
    p = (pred["y_prob_cal"] if "y_prob_cal" in pred else pred["y_prob"]).astype(float)

    src = df["sources"].to_numpy()[te]
    lab_all = df["label"].to_numpy()
    src_all = df["sources"].to_numpy()

    # (a) class composition per source (whole dataset)
    comp = {}
    for s in np.unique(src_all):
        m = src_all == s
        comp[str(s)] = {"n": int(m.sum()),
                        "active_rate": round(float(lab_all[m].mean()), 4)}
    overall_rate = round(float(lab_all.mean()), 4)

    # (c) decisive test: AUC restricted to each source in the HOLD-OUT test
    per_source = {}
    for s in np.unique(src):
        m = src == s
        if m.sum() < 30 or len(np.unique(y[m])) < 2:
            per_source[str(s)] = {"n_test": int(m.sum()), "note": "too few / single-class"}
            continue
        per_source[str(s)] = {
            "n_test": int(m.sum()),
            "active_rate": round(float(y[m].mean()), 4),
            "roc_auc": round(float(roc_auc_score(y[m], p[m])), 4),
            "pr_auc": round(float(average_precision_score(y[m], p[m])), 4),
        }

    overall_auc = round(float(roc_auc_score(y, p)), 4)
    # judge on the SUBSTANTIVE sources only (>=500 test compounds); tiny mixed
    # groups are too noisy to compare and would distort the spread.
    aucs = [v["roc_auc"] for v in per_source.values()
            if "roc_auc" in v and v["n_test"] >= 500]
    spread = round(float(max(aucs) - min(aucs)), 4) if len(aucs) > 1 else 0.0

    out = {
        "question": "Is the model learning a source/assay artefact instead of biology?",
        "overall_active_rate": overall_rate,
        "overall_test_roc_auc": overall_auc,
        "class_composition_by_source": comp,
        "test_auc_by_source": per_source,
        "auc_spread_across_major_sources": spread,
        "verdict": ("No source/assay artefact: restricting the hold-out to either "
                    f"major source leaves ROC-AUC essentially unchanged (spread {spread} "
                    "<= 0.03), despite differing per-source active rates -- the model "
                    "discriminates activity WITHIN each source, i.e. it learns biology, "
                    "not the source." if spread <= 0.03 else
                    f"AUC varies across major sources (spread {spread}); discuss."),
    }
    save_json(out, os.path.join(cfg.dir_metrics, "assay_source_bias.json"))

    print("\n================ DEEPENTXAI-12 (assay/source bias) ================")
    print(f"  overall active rate {overall_rate} | overall test AUC {overall_auc}")
    print("  class composition by source:")
    for s, v in comp.items():
        print(f"    {s:16s} n={v['n']:6d}  active_rate={v['active_rate']}")
    print("  hold-out AUC restricted to each source:")
    for s, v in per_source.items():
        if "roc_auc" in v:
            print(f"    {s:16s} n={v['n_test']:5d}  AUC={v['roc_auc']}  (active {v['active_rate']})")
    print(f"  AUC spread across sources: {spread}  ->  {out['verdict']}")


if __name__ == "__main__":
    main()


## `13_imbalanced_evaluation.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 13 | Imbalanced-Scenario Evaluation   (reviewer R1.2)
=================================================================================
Reviewer concern: the benchmark is near-balanced; show the model still works on a
highly UNBALANCED, real-world-like distribution (few actives, many inactives).

This stage does NOT retrain and does NOT change the headline accuracy. It takes the
trained headline model's hold-out predictions and stress-tests them at increasing
imbalance by keeping ALL inactives and subsampling actives down to ratios up to
1:100 (active:inactive). For each ratio it reports the metrics that are meaningful
under imbalance -- ROC-AUC (prevalence-invariant), PR-AUC, MCC, and the virtual-
screening Enrichment Factor -- averaged over several random subsamples.

Scientific point: ROC-AUC is provably invariant to class prevalence, so the model's
ability to DISCRIMINATE actives from inactives does not degrade as the data becomes
imbalanced; PR-AUC falls (as it must, since the positive base-rate falls) but the
Enrichment Factor shows the ranking stays highly useful for screening.

Consumes : 03_Results/05_Predictions/ensemble_test_predictions.npz (from stage 04)
Outputs  : 03_Results/02_Metrics/imbalanced_evaluation.json
           03_Results/01_Figures/imbalance_robustness.{png,pdf}
Run      : ~/miniconda3/envs/ENT/bin/python 13_imbalanced_evaluation.py
=================================================================================
"""
from __future__ import annotations
import os, sys
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import get_logger, save_json


def ef_at(y, p, frac):
    order = np.argsort(-p)
    k = max(1, int(round(frac * len(y))))
    hit = y[order][:k].mean()
    base = y.mean()
    return float(hit / base) if base > 0 else float("nan")


def main() -> None:
    cfg = Config.load()
    log = get_logger("NB13", cfg.dir_logs)
    from sklearn.metrics import (roc_auc_score, average_precision_score,
                                 matthews_corrcoef)

    pred = np.load(os.path.join(cfg.dir_predictions, "ensemble_test_predictions.npz"))
    y = pred["y_true"].astype(int)
    p = (pred["y_prob_cal"] if "y_prob_cal" in pred else pred["y_prob"]).astype(float)
    thr = float(pred["threshold"]) if "threshold" in pred else 0.5

    act = np.where(y == 1)[0]
    inact = np.where(y == 0)[0]
    n_act, n_inact = len(act), len(inact)
    rng = np.random.RandomState(cfg.seed)

    ratios = [(1, 1), (1, 2), (1, 5), (1, 10), (1, 25), (1, 50), (1, 100)]
    rows = []
    for ra, ri in ratios:
        # keep all inactives; subsample actives so that active:inactive = ra:ri
        target_act = int(round(n_inact * ra / ri))
        if target_act < 20:                 # keep enough positives to be meaningful
            continue
        target_act = min(target_act, n_act)
        aucs, prs, mccs, ef1, ef01 = [], [], [], [], []
        for rep in range(10):
            sel_act = rng.choice(act, size=target_act, replace=False)
            idx = np.concatenate([sel_act, inact])
            yy, pp = y[idx], p[idx]
            aucs.append(roc_auc_score(yy, pp))
            prs.append(average_precision_score(yy, pp))
            mccs.append(matthews_corrcoef(yy, (pp >= thr).astype(int)))
            ef1.append(ef_at(yy, pp, 0.01))
            ef01.append(ef_at(yy, pp, 0.001))
        rows.append({
            "ratio": f"1:{round(ri/ra)}",
            "active_rate": round(target_act / (target_act + n_inact), 4),
            "n_active": int(target_act), "n_inactive": int(n_inact),
            "roc_auc": round(float(np.mean(aucs)), 4),
            "pr_auc": round(float(np.mean(prs)), 4),
            "mcc": round(float(np.mean(mccs)), 4),
            "EF@1%": round(float(np.mean(ef1)), 2),
            "EF@0.1%": round(float(np.mean(ef01)), 2),
        })

    out = {
        "note": ("Trained headline model, hold-out predictions, no retraining. "
                 "Inactives all kept; actives subsampled to raise imbalance. "
                 "ROC-AUC is prevalence-invariant; EF shows screening utility."),
        "natural_test_balance": {"n_active": n_act, "n_inactive": n_inact,
                                 "active_rate": round(n_act / (n_act + n_inact), 4)},
        "by_ratio": rows,
    }
    save_json(out, os.path.join(cfg.dir_metrics, "imbalanced_evaluation.json"))

    # figure: ROC-AUC flat, PR-AUC declines, EF strong across imbalance
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    xr = [r["active_rate"] for r in rows]
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    ax.plot(xr, [r["roc_auc"] for r in rows], "-o", label="ROC-AUC", color="#4c72b0")
    ax.plot(xr, [r["pr_auc"] for r in rows], "-s", label="PR-AUC", color="#dd8452")
    ax.set_xscale("log")
    ax.set_xlabel("positive (active) rate  — more imbalanced →")
    ax.set_ylabel("area under curve"); ax.set_ylim(0, 1.02)
    ax.invert_xaxis(); ax.legend(loc="lower left", fontsize=9)
    ax2 = ax.twinx()
    ax2.plot(xr, [r["EF@1%"] for r in rows], "-^", label="EF@1%", color="#55a868")
    ax2.set_ylabel("Enrichment Factor @1%")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(cfg.dir_figures, f"F09_imbalance_robustness.{ext}"), dpi=300)
    plt.close(fig)

    print("\n================ DEEPENTXAI-13 (imbalanced evaluation) ================")
    print(f"  {'ratio':>8} {'act-rate':>9} {'ROC-AUC':>8} {'PR-AUC':>7} {'MCC':>6} {'EF@1%':>7} {'EF@0.1%':>8}")
    for r in rows:
        print(f"  {r['ratio']:>8} {r['active_rate']:>9} {r['roc_auc']:>8} "
              f"{r['pr_auc']:>7} {r['mcc']:>6} {r['EF@1%']:>7} {r['EF@0.1%']:>8}")
    print("  -> ROC-AUC stays ~flat (prevalence-invariant); ranking stays strongly enriched.")


if __name__ == "__main__":
    main()


## `14_feature_stability.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 14 | Feature-Importance Stability   (reviewer R2.9)
=================================================================================
Reviewer concern: interpretability claims need the STABILITY of feature rankings
across resamples / folds, otherwise the top-feature list may be noise.

Two complementary stability analyses (no retraining of the headline model, so
accuracy is untouched):

  (A) Bootstrap stability -- resample the hold-out set B times, recompute RDKit
      permutation importance each time, and report per-descriptor mean +/- SD,
      a 95% percentile CI, and how often each descriptor stays in the Top-10
      (selection frequency). Stable descriptors have high mean and high frequency.

  (B) Across-fold agreement -- run permutation importance under each of the 5
      ensemble fold models and report the mean pairwise Spearman rank correlation
      of the descriptor rankings. High correlation = the ranking is not fold-luck.

Consumes : 03_Results/03_Model/deepentxai_best.keras + ensemble_fold{0..4}.keras
Outputs  : 03_Results/02_Metrics/feature_stability.json
           03_Results/01_Figures/feature_stability_top15.{png,pdf}
Run      : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 14_feature_stability.py
=================================================================================
"""
from __future__ import annotations
import os, sys, json
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.train import load_matrices, slice_inputs
from deepentxai import explain as X_

B_BOOT = 20          # bootstrap resamples
SUBSAMPLE = 2500     # compounds per resample (enough for a stable ranking, keeps CPU sane)
N_REPEATS = 2


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB14", cfg.dir_logs)
    import keras
    from scipy.stats import spearmanr

    X, y, split = load_matrices(cfg)
    te = split["test_idx"]
    names = json.load(open(os.path.join(cfg.dir_features_selected,
                      "selected_rdkit_features.json")))["selected_rdkit_features"]
    rng = np.random.RandomState(cfg.seed)

    # ---- (A) bootstrap stability on the headline model ----------------------
    best = keras.models.load_model(os.path.join(cfg.dir_models, "deepentxai_best.keras"))
    cols = {n: [] for n in names}
    for b in range(B_BOOT):
        idx = rng.choice(te, size=min(SUBSAMPLE, len(te)), replace=True)
        Xb = slice_inputs(X, idx); yb = y[idx]
        imp = X_.permutation_descriptor(best, Xb, yb, names, n_repeats=N_REPEATS, seed=cfg.seed + b)
        d = dict(zip(imp["descriptor"], imp["importance"]))
        for n in names:
            cols[n].append(float(d.get(n, 0.0)))
        log.info(f"bootstrap {b+1}/{B_BOOT} done")

    arr = {n: np.array(v) for n, v in cols.items()}
    mean_imp = {n: float(a.mean()) for n, a in arr.items()}
    order = sorted(names, key=lambda n: mean_imp[n], reverse=True)
    # per-bootstrap top-10 sets -> selection frequency
    boot_rank = np.argsort(-np.column_stack([arr[n] for n in names]), axis=1)  # B x P
    top10_freq = {n: 0 for n in names}
    for b in range(B_BOOT):
        top10 = set(np.array(names)[boot_rank[b, :10]])
        for n in top10:
            top10_freq[n] += 1
    stability = []
    for n in order[:15]:
        a = arr[n]
        stability.append({
            "descriptor": n,
            "mean_importance": round(float(a.mean()), 5),
            "sd": round(float(a.std()), 5),
            "ci95": [round(float(np.percentile(a, 2.5)), 5),
                     round(float(np.percentile(a, 97.5)), 5)],
            "top10_frequency": round(top10_freq[n] / B_BOOT, 2),
        })

    keras.backend.clear_session()

    # ---- (B) across-fold rank agreement -------------------------------------
    fold_ranks = []
    for f in range(int(cfg["split"]["n_folds"])):
        fp = os.path.join(cfg.dir_models, f"ensemble_fold{f}.keras")
        if not os.path.exists(fp):
            continue
        m = keras.models.load_model(fp)
        idx = rng.choice(te, size=min(SUBSAMPLE, len(te)), replace=False)
        imp = X_.permutation_descriptor(m, slice_inputs(X, idx), y[idx], names,
                                        n_repeats=1, seed=cfg.seed)
        d = dict(zip(imp["descriptor"], imp["importance"]))
        fold_ranks.append([d.get(n, 0.0) for n in names])
        keras.backend.clear_session()
        log.info(f"fold {f} importance done")
    corrs = []
    for i in range(len(fold_ranks)):
        for j in range(i + 1, len(fold_ranks)):
            corrs.append(spearmanr(fold_ranks[i], fold_ranks[j]).correlation)
    mean_spearman = round(float(np.mean(corrs)), 3) if corrs else float("nan")

    out = {
        "bootstrap": {"n_resamples": B_BOOT, "subsample": SUBSAMPLE,
                      "top15_stable_descriptors": stability},
        "across_fold": {"n_folds": len(fold_ranks),
                        "mean_pairwise_spearman": mean_spearman},
        "verdict": (f"TOP features are stable (recur in >= "
                    f"{int(100*min(s['top10_frequency'] for s in stability[:5]))}% of bootstraps), "
                    f"but the FULL 100-descriptor ranking has low cross-fold correlation "
                    f"(Spearman {mean_spearman}) because most descriptors carry near-zero "
                    "importance and their order is unstable. Interpretability claims are "
                    "restricted to the stably-identified top features."),
    }
    save_json(out, os.path.join(cfg.dir_metrics, "feature_stability.json"))

    # figure: top-15 mean importance with SD error bars
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    top = stability[::-1]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh([s["descriptor"] for s in top], [s["mean_importance"] for s in top],
            xerr=[s["sd"] for s in top], color="#4c72b0", edgecolor="black", capsize=2)
    ax.set_xlabel("permutation importance (mean ± SD over bootstraps)")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(cfg.dir_figures, f"F10_feature_stability_top15.{ext}"), dpi=300)
    plt.close(fig)

    print("\n================ DEEPENTXAI-14 (feature stability) ================")
    print(f"  across-fold mean pairwise Spearman: {mean_spearman}")
    print(f"  {'descriptor':22s} {'mean':>8} {'sd':>7} {'top10-freq':>10}")
    for s in stability[:10]:
        print(f"  {s['descriptor']:22s} {s['mean_importance']:>8.4f} {s['sd']:>7.4f} {s['top10_frequency']:>10}")


if __name__ == "__main__":
    main()


## `15_external_cross_source.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | 15 | External Cross-Source Validation   (reviewer R2.3)
=================================================================================
Reviewer concern: no external dataset evaluation; high accuracy needs external
validation to rule out overfitting / assay-specific bias.

This is the strongest external test available in-house: TRAIN only on ChEMBL
compounds, TEST only on PubChem compounds (a different database with different
assays and submitters), and vice-versa. It is leakage-free by construction --
feature scaling is fit on the training source ONLY, and the model never sees a
single test-source compound during training. The 505 mixed-provenance compounds
are dropped so the two sets are provenance-disjoint.

This does NOT alter the headline model or its reported accuracy; it is an extra,
harder generalisation test. External AUC is expected to be a little below the
internal scaffold-disjoint AUC (a different chemical space) and still demonstrates
real transfer.

Consumes : 02_Data/03_Features_Raw/features.npz, 04_Features_Selected/rdkit_selected.npz
           02_Data/02_Processed/labelled.csv (sources), 03_Model/best_hparams.json
Outputs  : 03_Results/02_Metrics/external_cross_source.json
Run      : DEEPENT_THREADS=8 ~/miniconda3/envs/ENT/bin/python 15_external_cross_source.py
=================================================================================
"""
from __future__ import annotations
import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config
from deepentxai.utils import set_seed, get_logger, save_json
from deepentxai.model import INPUT_ORDER
from deepentxai.train import train_model, slice_inputs
from deepentxai.evaluate import compute_metrics


def build_X(feats, rsel, scaler_fit_idx):
    """Binary blocks raw; rdkit + chemberta scaled on the TRAIN-SOURCE rows only."""
    from sklearn.preprocessing import StandardScaler
    rk = rsel["X_rdkit_selected"].astype("float32")
    cb = feats["X_chemberta"].astype("float32")
    s_rk = StandardScaler().fit(rk[scaler_fit_idx])
    s_cb = StandardScaler().fit(cb[scaler_fit_idx])
    return {"morgan": feats["X_morgan"].astype("float32"),
            "rdkit": s_rk.transform(rk).astype("float32"),
            "maccs": feats["X_maccs"].astype("float32"),
            "chemberta": s_cb.transform(cb).astype("float32")}


def run_direction(cfg, feats, rsel, y, src, train_src, test_src, best, log):
    import keras
    tr_all = np.where(src == train_src)[0]
    te = np.where(src == test_src)[0]
    rng = np.random.RandomState(cfg.seed)
    rng.shuffle(tr_all)
    n_val = int(0.15 * len(tr_all))
    va, tr = tr_all[:n_val], tr_all[n_val:]

    X = build_X(feats, rsel, tr)                      # scale on TRAIN source rows only
    dims = {m: X[m].shape[1] for m in INPUT_ORDER}
    set_seed(cfg.seed)
    model, _ = train_model(X, y, dims, best, cfg, tr, va,
                           epochs=cfg["train"]["epochs"], patience=8, verbose=0)
    p = model.predict(slice_inputs(X, te), verbose=0).ravel()
    keras.backend.clear_session()
    from sklearn.metrics import roc_auc_score
    m = compute_metrics(y[te], p, threshold=0.5)
    m["roc_auc"] = float(roc_auc_score(y[te], p))
    m["n_train"] = int(len(tr)); m["n_test"] = int(len(te))
    m["train_source"] = train_src; m["test_source"] = test_src
    log.info(f"{train_src}->{test_src}: AUC {m['roc_auc']:.4f} acc {m['accuracy']:.4f} "
             f"(train {len(tr)} / test {len(te)})")
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()}


def main() -> None:
    cfg = Config.load()
    set_seed(cfg.seed)
    log = get_logger("NB15", cfg.dir_logs)

    feats = np.load(os.path.join(cfg.dir_features_raw, "features.npz"), allow_pickle=True)
    rsel = np.load(os.path.join(cfg.dir_features_selected, "rdkit_selected.npz"))
    y = feats["y"].astype(int)
    src = pd.read_csv(os.path.join(cfg.dir_processed, "labelled.csv"))["sources"].to_numpy()
    assert len(src) == len(y), "sources/features misalignment"
    best = {**cfg["model"], **json.load(open(
        os.path.join(cfg.dir_models, "best_hparams.json")))["best_params"]}

    results = {}
    results["ChEMBL_to_PubChem"] = run_direction(cfg, feats, rsel, y, src, "ChEMBL", "PubChem", best, log)
    results["PubChem_to_ChEMBL"] = run_direction(cfg, feats, rsel, y, src, "PubChem", "ChEMBL", best, log)

    aucs = [v["roc_auc"] for v in results.values()]
    out = {
        "design": ("Provenance-disjoint external validation. Train on one database, "
                   "test on the other. Scaling fit on the training source only; the "
                   "model never trains on any test-source compound. 505 mixed compounds "
                   "dropped."),
        "internal_scaffold_disjoint_auc": 0.901,
        "external_directions": results,
        "external_mean_auc": round(float(np.mean(aucs)), 4),
        "verdict": (f"Above-chance but MODERATE cross-database transfer: external mean "
                    f"ROC-AUC {round(float(np.mean(aucs)),3)} (well above random 0.5, but "
                    "below the internal scaffold-disjoint 0.901). This indicates a genuine "
                    "domain shift between ChEMBL and PubChem chemical spaces / activity "
                    "definitions. Reported transparently as a limitation; the model learns "
                    "real transferable signal but is not database-agnostic."),
    }
    save_json(out, os.path.join(cfg.dir_metrics, "external_cross_source.json"))

    print("\n================ DEEPENTXAI-15 (external cross-source) ================")
    for k, v in results.items():
        print(f"  {k:20s} AUC {v['roc_auc']}  acc {v['accuracy']}  "
              f"MCC {v['mcc']}  (train {v['n_train']} / test {v['n_test']})")
    print(f"  external mean AUC {out['external_mean_auc']}  (internal 0.901)")
    print(f"  -> {out['verdict']}")


if __name__ == "__main__":
    main()


## `make_manuscript_figures.py`


In [ ]:
#!/usr/bin/env python3
"""
=================================================================================
DEEPENTXAI_FINAL | Manuscript figure & table assembler
=================================================================================
Builds a numbered, submission-ready set under 03_Results/01_Figures/Manuscript/:

  * Tables A-F from REVISION_RESPONSE.md rendered as titleless figure images
    (T01..T06), with dynamic numbers read from the metrics JSONs.
  * All pipeline plots copied in under a manuscript figure number (F01..F12).
  * FIGURES.txt -- the numbered caption manifest for every item.

Non-destructive: the canonical pipeline figures in 01_Figures/ are left untouched,
so re-running any stage still works; this only assembles the numbered copies.

Run : ~/miniconda3/envs/ENT/bin/python make_manuscript_figures.py
=================================================================================
"""
from __future__ import annotations
import os, sys, json, shutil

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from deepentxai.config import Config

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ---- table image renderer ---------------------------------------------------
def render_table(rows, path, col_w=None, fontsize=11, header=True):
    """rows: list of lists (first row = header). Titleless clean table image."""
    ncol = len(rows[0]); nrow = len(rows)
    fig_w = max(6, sum(col_w) if col_w else 2.1 * ncol)
    fig_h = 0.5 * nrow + 0.4
    fig, ax = plt.subplots(figsize=(fig_w, fig_h)); ax.axis("off")
    tbl = ax.table(cellText=rows, cellLoc="center", loc="center",
                   colWidths=[w / sum(col_w) for w in col_w] if col_w else None)
    tbl.auto_set_font_size(False); tbl.set_fontsize(fontsize); tbl.scale(1, 1.4)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#bbbbbb")
        if header and r == 0:
            cell.set_facecolor("#4c72b0"); cell.set_text_props(color="white", weight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#f3f5f9")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(f"{path}.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig)


def main():
    cfg = Config.load()
    figdir = cfg.dir_figures
    M = cfg.dir_metrics
    out = figdir            # numbered set lives directly in 01_Figures (no subfolder)

    def load(name):
        p = os.path.join(M, name)
        return json.load(open(p)) if os.path.exists(p) else {}

    imb = load("imbalanced_evaluation.json")
    ext = load("external_cross_source.json")
    assay = load("assay_source_bias.json")
    stab = load("feature_stability.json")

    # ---- Table A : architecture --------------------------------------------
    A = [["Stage", "Layer", "Configuration", "Output"],
         ["Input x4", "Morgan / RDKit / MACCS / ChemBERTa", "2048 / 100 / 167 / 768", "per-branch"],
         ["Encoders x4", "Dense+BatchNorm+Dropout", "128, GELU, drop 0.25", "4 x 128"],
         ["Fusion", "Concatenate+BatchNorm", "-", "512"],
         ["To-sequence", "Dense -> Reshape", "512 -> (64x8)", "(64, 8)"],
         ["Conv x2", "residual Conv1D", "32 filters, k=3, GELU", "(64, 32)"],
         ["Attention", "squeeze-excite", "-", "(64, 32)"],
         ["Pool", "MaxPooling1D", "pool 2", "(32, 32)"],
         ["Recurrent", "Bidirectional LSTM", "64 units (->128)", "128"],
         ["Head", "res-Dense->Drop->Dense", "64, GELU", "64"],
         ["Output", "Dense", "1, sigmoid", "P(active)"]]
    render_table(A, os.path.join(out, "T01_model_architecture"),
                 col_w=[1.4, 3.2, 2.6, 1.4], fontsize=10)

    # ---- Table B : feature accounting --------------------------------------
    B = [["Tool / block", "Computed", "Removed", "Kept", "Selection"],
         ["Morgan (r=2)", "2048", "0", "2048", "-"],
         ["RDKit descriptors", "208", "108", "100", "var -> MI -> RFE (train-only)"],
         ["MACCS keys", "167", "0", "167", "-"],
         ["ChemBERTa embedding", "768", "0", "768", "-"],
         ["TOTAL model input", "", "", "3083", "no PCA"]]
    render_table(B, os.path.join(out, "T02_feature_accounting"),
                 col_w=[2.2, 1.3, 1.3, 1.1, 3.4], fontsize=10)

    # ---- Table C : performance vs imbalance --------------------------------
    C = [["active:inactive", "active rate", "ROC-AUC", "PR-AUC", "MCC", "EF@1%", "EF@0.1%"]]
    for r in imb.get("by_ratio", []):
        C.append([r["ratio"], f"{r['active_rate']}", f"{r['roc_auc']}", f"{r['pr_auc']}",
                  f"{r['mcc']}", f"{r['EF@1%']}", f"{r['EF@0.1%']}"])
    render_table(C, os.path.join(out, "T03_performance_vs_imbalance"),
                 col_w=[1.8, 1.3, 1.2, 1.1, 1.0, 1.0, 1.2], fontsize=10)

    # ---- Table D : external cross-source -----------------------------------
    D = [["Train -> Test", "n train", "n test", "ROC-AUC", "Accuracy", "MCC"]]
    for k, v in ext.get("external_directions", {}).items():
        D.append([k.replace("_", " "), f"{v['n_train']}", f"{v['n_test']}",
                  f"{v['roc_auc']}", f"{v['accuracy']}", f"{v['mcc']}"])
    D.append(["Mean external", "", "", f"{ext.get('external_mean_auc','')}", "", ""])
    D.append(["Internal (scaffold-disjoint)", "", "", "0.901", "0.840", "0.655"])
    render_table(D, os.path.join(out, "T04_external_cross_source"),
                 col_w=[2.6, 1.2, 1.2, 1.2, 1.2, 1.0], fontsize=10)

    # ---- Table E : assay/source bias ---------------------------------------
    E = [["Restriction", "n (test)", "active rate", "ROC-AUC"]]
    for s, v in assay.get("test_auc_by_source", {}).items():
        if "roc_auc" in v:
            E.append([s, f"{v['n_test']}", f"{v['active_rate']}", f"{v['roc_auc']}"])
    E.append(["AUC spread (major)", "", "", f"{assay.get('auc_spread_across_major_sources','')}"])
    render_table(E, os.path.join(out, "T05_assay_source_bias"),
                 col_w=[2.4, 1.3, 1.5, 1.3], fontsize=10)

    # ---- Table F : feature stability ---------------------------------------
    F = [["Descriptor", "mean imp.", "SD", "Top-10 freq."]]
    for s in stab.get("bootstrap", {}).get("top15_stable_descriptors", [])[:8]:
        F.append([s["descriptor"], f"{s['mean_importance']}", f"{s['sd']}", f"{s['top10_frequency']}"])
    sp = stab.get("across_fold", {}).get("mean_pairwise_spearman", "")
    F.append([f"across-fold Spearman = {sp}", "", "", ""])
    render_table(F, os.path.join(out, "T06_feature_stability"),
                 col_w=[3.0, 1.4, 1.1, 1.5], fontsize=10)

    # ---- plots are written with their F## number natively by each stage -----
    PLOTS = [
        ("F01", "F01_dataset_overview.png",     "Dataset composition (49,093 activity-gap compounds; class balance and source split)."),
        ("F02", "F02_roc_curve.png",            "ROC curve on the scaffold-disjoint hold-out (ROC-AUC 0.901)."),
        ("F03", "F03_pr_curve.png",             "Precision-recall curve on the hold-out (PR-AUC 0.870)."),
        ("F04", "F04_calibration_curve.png",    "Reliability (calibration) curve after isotonic calibration."),
        ("F05", "F05_confusion_matrix.png",     "Confusion matrix at the operating threshold. [Supplementary S1]"),
        ("F06", "F06_learning_curve.png",       "Training history (loss/metric vs epoch). Not a substitute for generalisation."),
        ("F07", "F07_risk_coverage.png",        "Conformal risk-coverage curve: accuracy vs fraction of compounds called (98.1% at 22% coverage)."),
        ("F08", "F08_enrichment_curve.png",     "Retrospective virtual-screen accumulation curve (top-1% = 100% true actives)."),
        ("F09", "F09_imbalance_robustness.png", "Performance vs class imbalance: ROC-AUC flat, Enrichment Factor rises (R1.2)."),
        ("F10", "F10_feature_stability_top15.png","Bootstrap feature-importance stability, top-15 RDKit descriptors, mean +/- SD (R2.9)."),
        ("F11", "F11_regression_scatter.png",   "Secondary task: predicted vs measured E. coli pMIC (R2 0.595)."),
        ("F12", "F12_summary.png",              "Summary performance panel."),
    ]
    copied = [(num, fn, cap) for num, fn, cap in PLOTS if os.path.exists(os.path.join(out, fn))]

    TABLES = [
        ("T01", "T01_model_architecture.png",   "DeepEntXAI architecture (layers, shapes; 1,276,933 parameters)."),
        ("T02", "T02_feature_accounting.png",   "Feature accounting: 2048+100+167+768 = 3083 features, no PCA (R2.4)."),
        ("T03", "T03_performance_vs_imbalance.png","Performance across class-imbalance ratios; EF@0.1% up to 41.9x at 1:50 (R1.2)."),
        ("T04", "T04_external_cross_source.png", "Provenance-disjoint external validation, train/test across databases (R2.3)."),
        ("T05", "T05_assay_source_bias.png",    "Hold-out ROC-AUC restricted to each source; spread 0.021 -> no assay artefact (R2.6)."),
        ("T06", "T06_feature_stability.png",    "Bootstrap + across-fold feature-importance stability (R2.9)."),
    ]

    # ---- FIGURES.txt manifest ----------------------------------------------
    lines = ["DEEPENTXAI - Manuscript figure & table index",
             "=" * 60, "",
             "FIGURES (plots) -- titleless; caption is the sole title.", ""]
    for num, fn, cap in copied:
        lines.append(f"Figure {int(num[1:])}  ({fn})")
        lines.append(f"    {cap}"); lines.append("")
    lines += ["", "TABLES (rendered as figure images).", ""]
    for i, (num, fn, cap) in enumerate(TABLES, 1):
        lines.append(f"Table {chr(64+i)}  ({fn})")
        lines.append(f"    {cap}"); lines.append("")
    with open(os.path.join(out, "FIGURES.txt"), "w") as fh:
        fh.write("\n".join(lines))

    print("\n================ manuscript figures assembled ================")
    print(f"  output dir : {out}")
    print(f"  tables     : T01..T06 (rendered)")
    print(f"  figures    : {len(copied)} plots copied (F01..F{len(copied):02d})")
    print(f"  manifest   : FIGURES.txt")


if __name__ == "__main__":
    main()
